# ARC-AGI-2 Program No. 063
## Exact Program 024 Candidate Generation with Conservative Cross-Symmetry Second-Guess Guard

**Runtime ID guard fix:** the run-plan identity check now references `PROGRAM_NO` rather than a copied numeric literal, preventing stale-version mismatches.

This notebook preserves the exact Program 024 loader, decoder, solver, training recipe, stochastic CUDA backward path, and Program 052 dual-runtime compatibility. It adds **no external model, no extra training pass, and no extra GPU inference**.

The only experimental change is final submission assembly: attempt 1 remains the exact KGMon top-1 candidate. Attempt 2 is changed only when a nearby candidate (KGMon top-4 and full-probability top-4) has strictly broader geometric-augmentation support than the original KGMon rank-2 candidate. This is intended to reduce duplicate-beam / single-symmetry overconfidence while preserving the trusted first guess.

The notebook also writes a shadow baseline submission using the untouched KGMon top-2 ranking and prints both local public scores, so the reranking effect is visible without another GPU run. The final Kaggle file remains `/kaggle/working/submission.json`.


In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROGRAM_NO = 63
BASE_SEED = 42
PYTHON_HASH_SEED = "0"
SEED_CONTRACT = {
    "base_seed": BASE_SEED,
    "python_hash_seed": PYTHON_HASH_SEED,
    "worker_init_reset": True,
    "task_dequeue_reset": True,
    "deterministic_algorithms_enabled": False,
    "backend_policy_changed": False,
}
NOTEBOOK_START_TIME = time.time()
global_end_time = NOTEBOOK_START_TIME + 12 * 3600 - 600
MODEL_PATH = Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
RUNTIME_HINTS = [
    Path("/kaggle/usr/lib/notebooks/mirzamilanfarabi/pip_install_unsloth_flash_patch"),
    Path("/kaggle/usr/lib/notebooks/sorokin/pip_install_unsloth_flash_patch"),
]
PROFILE_ROOT = Path("/kaggle/working/program063_runtime_profiles")
WORK_ROOT = Path("/kaggle/working")

NATIVE_BINARY_ANCHORS = {
    "torch", "triton", "torchvision", "torchaudio", "tokenizers", "safetensors",
    "numpy", "scipy", "pandas", "pyarrow", "sentencepiece", "google", "PIL",
}
GENERIC_SHARED_LIBRARY_ALLOWLIST = {"bitsandbytes"}


# Kaggle can expose either of two known, internally coherent Python package
# families under the mounted Unsloth runtime. Program 063 searches every
# mounted source root before GPU work. It prefers the Program 050 family that
# produced Private 29.31, and falls back only when that exact family is absent.
# Each selected distribution is still verified from its own METADATA/RECORD.
RUNTIME_FAMILIES = [
    (
        "program050_current",
        {
            "accelerate": {
                "version": "1.13.0",
                "official_wheel_filename": "accelerate-1.13.0-py3-none-any.whl",
                "official_wheel_sha256": "cf1a3efb96c18f7b152eb0fa7490f3710b19c3f395699358f08decca2b8b62e0",
            },
            "peft": {
                "version": "0.18.1",
                "official_wheel_filename": "peft-0.18.1-py3-none-any.whl",
                "official_wheel_sha256": "0bf06847a3551e3019fc58c440cffc9a6b73e6e2962c95b52e224f77bbdb50f1",
            },
            "bitsandbytes": {
                "version": "0.49.2",
                "official_wheel_filename": "bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl",
                "official_wheel_sha256": "54b771f06e1a3c73af5c7f16ccf0fc23a846052813d4b008d10cb6e017dd1c8c",
            },
        },
    ),
    (
        "program047_legacy",
        {
            "accelerate": {
                "version": "1.11.0",
                "official_wheel_filename": "accelerate-1.11.0-py3-none-any.whl",
                "official_wheel_sha256": "a628fa6beb069b8e549460fc449135d5bd8d73e7a11fd09f0bc9fc4ace7f06f1",
            },
            "peft": {
                "version": "0.17.1",
                "official_wheel_filename": "peft-0.17.1-py3-none-any.whl",
                "official_wheel_sha256": "3d129d64def3d74779c32a080d2567e5f7b674e77d546e3585138216d903f99e",
            },
            "bitsandbytes": {
                "version": "0.48.2",
                "official_wheel_filename": "bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl",
                "official_wheel_sha256": "cd289562cb7308ee2a707e6884fecca9bbbcfc9ec33a86df2a45e0779692c1a3",
            },
        },
    ),
]
PINNED_RUNTIME_DISTRIBUTIONS: dict[str, dict[str, str]] = {}



def env_flag(name: str) -> bool:
    value = os.getenv(name)
    return value is not None and value.strip().lower() in {"1", "true", "yes", "y", "on"}


def canonical_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_named_file(names: list[str]) -> Path:
    roots = [
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2"),
        Path("/kaggle/input/arc-prize-2026-arc-agi-2"),
    ]
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.is_file():
                return candidate.resolve()
    input_root = Path("/kaggle/input")
    for name in names:
        matches = sorted(input_root.rglob(name))
        if matches:
            return matches[0].resolve()
    raise FileNotFoundError(f"Could not resolve any of {names} below /kaggle/input")


def resolve_ptxas() -> Path:
    candidates = [
        Path("/usr/local/cuda/bin/ptxas"),
        Path("/usr/local/cuda-12.8/bin/ptxas"),
        Path("/usr/local/cuda-12.5/bin/ptxas"),
    ]
    discovered = shutil.which("ptxas")
    if discovered:
        candidates.append(Path(discovered))
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if resolved.is_file() and os.access(resolved, os.X_OK):
            return resolved
    raise FileNotFoundError("ptxas was not found; aborting before GPU model loading")


def discover_legacy_roots() -> list[Path]:
    candidates = list(RUNTIME_HINTS)
    notebook_root = Path("/kaggle/usr/lib/notebooks")
    if notebook_root.is_dir():
        candidates.extend(sorted(notebook_root.glob("**/pip_install_unsloth_flash_patch")))
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for unsloth_init in sorted(input_root.glob("**/unsloth/__init__.py")):
            candidates.append(unsloth_init.parent.parent)
    roots: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / "unsloth" / "__init__.py").is_file():
            roots.append(resolved)
    if not roots:
        raise FileNotFoundError(
            "PROGRAM063_LEGACY_RUNTIME_MISSING: no mounted Unsloth runtime root was found"
        )
    return roots


def read_metadata_name(dist_info: Path) -> str | None:
    metadata = dist_info / "METADATA"
    if not metadata.is_file():
        return None
    for line in metadata.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("Name:"):
            return line.split(":", 1)[1].strip()
    return None



def read_metadata_version(dist_info: Path) -> str | None:
    metadata = dist_info / "METADATA"
    if not metadata.is_file():
        return None
    for line in metadata.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return None


def distribution_index(
    source_root: Path,
    runtime_lock: dict[str, dict[str, str]],
    runtime_family: str,
) -> tuple[dict[str, Path], dict[str, str], dict[str, list[str]]]:
    catalog: dict[str, list[tuple[str, Path]]] = {}
    module_to_dist: dict[str, str] = {}
    for dist_info in sorted(source_root.glob("*.dist-info")):
        name = read_metadata_name(dist_info)
        version = read_metadata_version(dist_info)
        if not name or not version:
            continue
        canonical = canonical_name(name)
        catalog.setdefault(canonical, []).append((version, dist_info))
        top_level = dist_info / "top_level.txt"
        modules: list[str] = []
        if top_level.is_file():
            modules = [
                line.strip()
                for line in top_level.read_text(encoding="utf-8", errors="replace").splitlines()
                if line.strip()
            ]
        if not modules:
            record = dist_info / "RECORD"
            if record.is_file():
                with record.open("r", encoding="utf-8", errors="replace", newline="") as handle:
                    for row in csv.reader(handle):
                        if not row:
                            continue
                        parts = Path(row[0]).parts
                        first = parts[0] if parts else ""
                        if first and not first.endswith((".dist-info", ".data")):
                            module_name = first[:-3] if first.endswith(".py") else first
                            if module_name.isidentifier():
                                modules.append(module_name)
        for module_name in modules:
            module_to_dist.setdefault(module_name.split(".", 1)[0], canonical)

    available_versions = {
        canonical: sorted({version for version, _ in entries})
        for canonical, entries in catalog.items()
    }
    pinned_matches: dict[str, Path] = {}
    unavailable: dict[str, dict] = {}
    for canonical, lock in sorted(runtime_lock.items()):
        expected = lock["version"]
        entries = catalog.get(canonical, [])
        matches = [dist_info for version, dist_info in entries if version == expected]
        if len(matches) != 1:
            unavailable[canonical] = {
                "expected_version": expected,
                "available_versions": available_versions.get(canonical, []),
                "matching_dist_info_count": len(matches),
            }
        else:
            pinned_matches[canonical] = matches[0]
    if unavailable:
        raise RuntimeError(
            "PROGRAM063_RUNTIME_FAMILY_UNAVAILABLE="
            + json.dumps({
                "runtime_family": runtime_family,
                "source_root": str(source_root),
                "unavailable": unavailable,
                "expected_versions": {
                    name: lock["version"] for name, lock in sorted(runtime_lock.items())
                },
            }, sort_keys=True)
        )

    by_name: dict[str, Path] = {}
    for canonical, entries in catalog.items():
        if canonical in pinned_matches:
            by_name[canonical] = pinned_matches[canonical]
        else:
            by_name[canonical] = sorted(entries, key=lambda item: item[1].name)[-1][1]
    return by_name, module_to_dist, available_versions

def distribution_roots(source_root: Path, dist_info: Path) -> list[Path]:
    roots: set[Path] = set()
    top_level = dist_info / "top_level.txt"
    if top_level.is_file():
        for line in top_level.read_text(encoding="utf-8", errors="replace").splitlines():
            module = line.strip()
            if not module:
                continue
            for candidate in (source_root / module, source_root / f"{module}.py"):
                if candidate.exists():
                    roots.add(candidate)
    record = dist_info / "RECORD"
    if record.is_file():
        with record.open("r", encoding="utf-8", errors="replace", newline="") as handle:
            for row in csv.reader(handle):
                if not row:
                    continue
                rel = Path(row[0])
                if rel.is_absolute() or ".." in rel.parts or not rel.parts:
                    continue
                first = rel.parts[0]
                if first.endswith((".dist-info", ".data")):
                    continue
                candidate = source_root / first
                if candidate.exists():
                    roots.add(candidate)
    roots.add(dist_info)
    return sorted(roots, key=lambda path: path.name)



def _decode_record_sha256(value: str) -> bytes:
    padding = "=" * (-len(value) % 4)
    return base64.urlsafe_b64decode(value + padding)


def verify_distribution_record(
    source_root: Path,
    dist_name: str,
    by_name: dict[str, Path],
) -> dict:
    canonical = canonical_name(dist_name)
    lock = PINNED_RUNTIME_DISTRIBUTIONS[canonical]
    dist_info = by_name.get(canonical)
    if dist_info is None:
        raise RuntimeError(f"PROGRAM063_PINNED_DIST_INFO_MISSING={canonical}")
    actual_version = read_metadata_version(dist_info)
    if actual_version != lock["version"]:
        raise RuntimeError(
            "PROGRAM063_PINNED_VERSION_MISMATCH="
            + json.dumps({
                "distribution": canonical,
                "expected": lock["version"],
                "actual": actual_version,
                "dist_info": str(dist_info),
            }, sort_keys=True)
        )

    record = dist_info / "RECORD"
    metadata = dist_info / "METADATA"
    if not record.is_file() or not metadata.is_file():
        raise RuntimeError(f"PROGRAM063_PINNED_METADATA_INCOMPLETE={dist_info}")

    verified_entries: list[tuple[str, str, int]] = []
    skipped_external = 0
    unhashed_entries = 0
    with record.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for row in csv.reader(handle):
            if not row:
                continue
            rel_text = row[0]
            hash_spec = row[1] if len(row) > 1 else ""
            size_text = row[2] if len(row) > 2 else ""
            rel = Path(rel_text)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts:
                skipped_external += 1
                continue
            target = source_root / rel
            if not hash_spec:
                unhashed_entries += 1
                continue
            if "=" not in hash_spec:
                raise RuntimeError(f"PROGRAM063_RECORD_HASH_FORMAT_INVALID={canonical}:{rel_text}")
            algorithm, encoded = hash_spec.split("=", 1)
            if algorithm != "sha256":
                raise RuntimeError(f"PROGRAM063_RECORD_HASH_ALGORITHM_INVALID={canonical}:{algorithm}")
            if not target.is_file():
                raise RuntimeError(f"PROGRAM063_RECORD_FILE_MISSING={canonical}:{target}")
            actual_digest = hashlib.sha256(target.read_bytes()).digest()
            expected_digest = _decode_record_sha256(encoded)
            if actual_digest != expected_digest:
                raise RuntimeError(
                    "PROGRAM063_RECORD_FILE_HASH_MISMATCH="
                    + json.dumps({
                        "distribution": canonical,
                        "file": rel.as_posix(),
                        "expected": expected_digest.hex(),
                        "actual": actual_digest.hex(),
                    }, sort_keys=True)
                )
            actual_size = target.stat().st_size
            if size_text and actual_size != int(size_text):
                raise RuntimeError(
                    "PROGRAM063_RECORD_FILE_SIZE_MISMATCH="
                    + json.dumps({
                        "distribution": canonical,
                        "file": rel.as_posix(),
                        "expected": int(size_text),
                        "actual": actual_size,
                    }, sort_keys=True)
                )
            verified_entries.append((rel.as_posix(), actual_digest.hex(), actual_size))

    if not verified_entries:
        raise RuntimeError(f"PROGRAM063_RECORD_NO_VERIFIED_FILES={canonical}")
    tree_digest = hashlib.sha256()
    for rel_text, digest_hex, size in sorted(verified_entries):
        tree_digest.update(f"{rel_text}\0{digest_hex}\0{size}\n".encode("utf-8"))
    return {
        "distribution": canonical,
        "version": actual_version,
        "dist_info": dist_info.name,
        "metadata_sha256": sha256_file(metadata),
        "record_sha256": sha256_file(record),
        "verified_file_count": len(verified_entries),
        "verified_byte_count": sum(size for _, _, size in verified_entries),
        "record_tree_sha256": tree_digest.hexdigest(),
        "unhashed_record_entries": unhashed_entries,
        "skipped_external_record_entries": skipped_external,
        "official_wheel_filename_reference": lock["official_wheel_filename"],
        "official_wheel_sha256_reference": lock["official_wheel_sha256"],
    }


def verify_pinned_runtime(source_root: Path, by_name: dict[str, Path]) -> dict[str, dict]:
    return {
        canonical: verify_distribution_record(source_root, canonical, by_name)
        for canonical in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    }

def incompatible_extension(root: Path, dist_name: str) -> str | None:
    files = [root] if root.is_file() else list(root.rglob("*"))
    for path in files:
        if not path.is_file() or path.suffix != ".so":
            continue
        lower = path.name.lower()
        if re.search(r"cpython-3(?:9|10|11)-", lower):
            return str(path)
        if dist_name not in GENERIC_SHARED_LIBRARY_ALLOWLIST and "cpython-" in lower:
            return str(path)
    return None


def copy_distribution(
    source_root: Path,
    overlay_root: Path,
    dist_name: str,
    by_name: dict[str, Path],
) -> list[str]:
    canonical = canonical_name(dist_name)
    if canonical in NATIVE_BINARY_ANCHORS:
        raise RuntimeError(f"Refusing to overlay native binary anchor: {dist_name}")
    dist_info = by_name.get(canonical)
    if dist_info is None:
        raise FileNotFoundError(f"Legacy distribution not found: {dist_name}")
    copied: list[str] = []
    for source in distribution_roots(source_root, dist_info):
        bad = incompatible_extension(source, canonical)
        if bad:
            raise RuntimeError(f"Legacy distribution {dist_name} contains a Python-ABI extension: {bad}")
        destination = overlay_root / source.name
        if destination.exists():
            continue
        if source.is_dir():
            shutil.copytree(source, destination, symlinks=False)
        else:
            shutil.copy2(source, destination)
        copied.append(source.name)
    return copied


def sanitize_pythonpath(current: str, source_root: Path, overlay_root: Path) -> str:
    result = [str(overlay_root)]
    seen = {overlay_root.resolve()}
    for raw in current.split(os.pathsep):
        if not raw:
            continue
        try:
            resolved = Path(raw).resolve()
        except OSError:
            continue
        if resolved == source_root or source_root in resolved.parents or resolved in seen:
            continue
        seen.add(resolved)
        result.append(raw)
    return os.pathsep.join(result)


def patch_legacy_transformers(overlay_root: Path) -> list[str]:
    patches: list[str] = []
    table = overlay_root / "transformers" / "dependency_versions_table.py"
    if table.is_file():
        text = table.read_text(encoding="utf-8")
        updated = re.sub(
            r'(tokenizers>=0\.\d+(?:\.\d+)?,<)0\.22',
            r'\g<1>0.23',
            text,
        )
        if updated != text:
            table.write_text(updated, encoding="utf-8")
            patches.append("transformers:tokenizers<0.23")
    return patches



def install_qwen3_official_kv_cache_patch(overlay_root: Path) -> dict:
    '''Patch only the broken legacy Qwen3 cached-attention call site.

    The copied Unsloth 2025.9.7 source has one cached-inference call to
    ``flash_attn_func(Qnn, Knn, Vnn)`` even when FlashAttention is unavailable.
    Program 063 replaces that single call with an in-module helper matching the
    current Unsloth Qwen3 branch structure, while leaving the surrounding legacy
    RoPE, KV-cache storage, output projection, and Program 024 solver untouched.
    '''
    qwen3_path = overlay_root / "unsloth" / "models" / "qwen3.py"
    if not qwen3_path.is_file():
        raise FileNotFoundError(f"Program 063 Qwen3 source is missing: {qwen3_path}")

    marker = "# PROGRAM063_QWEN3_OFFICIAL_KV_CACHE_V1"
    expected_original_sha256 = "434edd417020c58f15cfa94c10a0e5abd63e95a9ec83276988b876e63942ae09"
    original = qwen3_path.read_text(encoding="utf-8")
    before_hash = hashlib.sha256(original.encode("utf-8")).hexdigest()
    installed = False
    replacements = 0

    if marker in original:
        patched = original
    else:
        if before_hash != expected_original_sha256:
            raise RuntimeError(
                "PROGRAM063_QWEN3_SOURCE_HASH_MISMATCH="
                + json.dumps({
                    "expected": expected_original_sha256,
                    "actual": before_hash,
                    "path": str(qwen3_path),
                }, sort_keys=True)
            )
        legacy_call = "flash_attn_func(Qnn, Knn, Vnn)"
        if original.count(legacy_call) != 1:
            raise RuntimeError(
                f"PROGRAM063_QWEN3_LEGACY_CALL_COUNT={original.count(legacy_call)}"
            )
        replacement = (
            "_program063_qwen3_cached_attention("
            "Qnn, Knn, Vnn, "
            "attention_mask=locals().get('attention_mask', None), "
            "sliding_window=getattr(self.config, 'sliding_window', None))"
        )
        body = original.replace(legacy_call, replacement, 1)
        replacements = 1
        helper = r'''

# PROGRAM063_QWEN3_OFFICIAL_KV_CACHE_V1
# B,S,H,D compatibility helper for the one legacy cached-inference call site.
def _program063_qwen3_cached_attention(
    q,
    k,
    v,
    *,
    attention_mask=None,
    sliding_window=None,
):
    import torch
    import torch.nn.functional as _program063_F

    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:
        raise RuntimeError(
            f"PROGRAM063_QWEN3_EXPECTED_BSHD got={q.shape},{k.shape},{v.shape}"
        )
    if q.shape[0] != k.shape[0] or k.shape[0] != v.shape[0]:
        raise RuntimeError("PROGRAM063_QWEN3_BATCH_MISMATCH")
    if k.shape[0] != v.shape[0] or k.shape[1] != v.shape[1] or k.shape[2:] != v.shape[2:]:
        raise RuntimeError("PROGRAM063_QWEN3_KV_MISMATCH")
    if q.shape[-1] != k.shape[-1]:
        raise RuntimeError("PROGRAM063_QWEN3_HEAD_DIM_MISMATCH")

    # The legacy call uses FlashAttention layout [B, S, H, D].
    q_t = q.transpose(1, 2)
    k_t = k.transpose(1, 2)
    v_t = v.transpose(1, 2)
    batch_size = q_t.shape[0]
    q_heads = q_t.shape[1]
    kv_heads = k_t.shape[1]
    if kv_heads <= 0 or q_heads % kv_heads != 0:
        raise RuntimeError(
            f"PROGRAM063_QWEN3_GQA_MISMATCH q_heads={q_heads} kv_heads={kv_heads}"
        )
    groups = q_heads // kv_heads

    # Match current Unsloth's cached Qwen3 window handling.
    if sliding_window is not None:
        try:
            window = int(sliding_window)
        except (TypeError, ValueError):
            window = 0
        if window > 0 and k_t.shape[-2] > window:
            start = k_t.shape[-2] - window
            k_t = k_t[:, :, start:, :]
            v_t = v_t[:, :, start:, :]
            if attention_mask is not None:
                attention_mask = attention_mask[..., start:]

    q_len = q_t.shape[-2]
    k_len = k_t.shape[-2]
    mask = attention_mask
    if mask is not None:
        if mask.dim() == 2:
            mask = mask[:, None, None, :].to(torch.bool)
        elif mask.dim() == 4 and mask.dtype != torch.bool:
            mask = mask.eq(0)
        elif mask.dim() not in (3, 4):
            raise RuntimeError(f"PROGRAM063_QWEN3_MASK_DIM={mask.dim()}")
        if mask.shape[-1] > k_len:
            mask = mask[..., -k_len:]
        elif mask.shape[-1] < k_len:
            raise RuntimeError(
                f"PROGRAM063_QWEN3_MASK_LENGTH mask={mask.shape[-1]} kv={k_len}"
            )

    is_causal = mask is None and q_len == k_len
    use_sdpa_gqa = groups != 1
    if (
        use_sdpa_gqa
        and isinstance(mask, torch.Tensor)
        and mask.dim() >= 3
        and mask.shape[0] > 1
    ):
        # Current Unsloth avoids SDPA-GQA drift for batched masked decode.
        use_sdpa_gqa = False

    # Current Unsloth uses explicit attention for batch one. It also expands
    # K/V whenever SDPA-GQA is unavailable or deliberately disabled.
    if batch_size == 1 or (groups != 1 and not use_sdpa_gqa):
        k_t = k_t.repeat_interleave(groups, dim=1)
        v_t = v_t.repeat_interleave(groups, dim=1)
        use_sdpa_gqa = False

    if batch_size == 1:
        scale = q_t.shape[-1] ** -0.5
        scores = torch.matmul(q_t * scale, k_t.transpose(-2, -1))
        probabilities = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_t.dtype)
        out = torch.matmul(probabilities, v_t)
    else:
        kwargs = {
            "attn_mask": mask,
            "dropout_p": 0.0,
            "is_causal": is_causal,
        }
        if use_sdpa_gqa:
            try:
                out = _program063_F.scaled_dot_product_attention(
                    q_t, k_t, v_t, enable_gqa=True, **kwargs
                )
            except TypeError:
                k_t = k_t.repeat_interleave(groups, dim=1)
                v_t = v_t.repeat_interleave(groups, dim=1)
                out = _program063_F.scaled_dot_product_attention(q_t, k_t, v_t, **kwargs)
        else:
            out = _program063_F.scaled_dot_product_attention(q_t, k_t, v_t, **kwargs)
    return out.transpose(1, 2).contiguous()

_program063_qwen3_cached_attention.__program063_backend__ = "official_style_kv_cache"
'''
        patched = body.rstrip() + "\n" + helper.lstrip("\n")
        compile(patched, str(qwen3_path), "exec")
        qwen3_path.write_text(patched, encoding="utf-8")
        installed = True

    final = qwen3_path.read_text(encoding="utf-8")
    if final.count(marker) != 1:
        raise RuntimeError(
            f"Program 063 expected one Qwen3 helper marker, found {final.count(marker)}"
        )
    if "def flash_attn_func(" in final and marker in final:
        raise RuntimeError("PROGRAM063_GENERIC_FLASH_WRAPPER_PRESENT")
    if final.count("flash_attn_func(Qnn, Knn, Vnn)") != 0:
        raise RuntimeError("PROGRAM063_LEGACY_FLASH_CALL_REMAINS")
    if final.count("_program063_qwen3_cached_attention(Qnn, Knn, Vnn") != 1:
        raise RuntimeError("PROGRAM063_PATCHED_CALL_COUNT_MISMATCH")
    compile(final, str(qwen3_path), "exec")
    return {
        "path": str(qwen3_path),
        "installed": installed,
        "before_sha256": before_hash,
        "after_sha256": hashlib.sha256(final.encode("utf-8")).hexdigest(),
        "expected_original_sha256": expected_original_sha256,
        "helper_marker_count": final.count(marker),
        "call_replacements": replacements,
        "legacy_flash_calls_after": final.count("flash_attn_func(Qnn, Knn, Vnn)"),
        "generic_flash_wrapper": False,
    }


def write_qwen3_official_kv_probe(path: Path, program_no: int) -> None:
    path.write_text(
        bootstrap_source(program_no)
        + r'''
import json
import torch
import torch.nn.functional as F
import unsloth.models.qwen3 as qwen3

fn = getattr(qwen3, "_program063_qwen3_cached_attention", None)
if not callable(fn):
    raise RuntimeError("PROGRAM063_QWEN3_OFFICIAL_HELPER_MISSING")
backend = getattr(fn, "__program063_backend__", None)
if backend != "official_style_kv_cache":
    raise RuntimeError(f"PROGRAM063_QWEN3_BACKEND={backend}")

torch.manual_seed(47047)
device = torch.device("cuda:0")
dtype = torch.bfloat16
reports = []


def explicit_reference(q, k, v, *, mask=None, sliding_window=None, batch_one=False):
    q_t = q.transpose(1, 2)
    k_t = k.transpose(1, 2)
    v_t = v.transpose(1, 2)
    if sliding_window is not None and k_t.shape[-2] > sliding_window:
        start = k_t.shape[-2] - sliding_window
        k_t = k_t[:, :, start:, :]
        v_t = v_t[:, :, start:, :]
        if mask is not None:
            mask = mask[..., start:]
    if mask is not None:
        if mask.dim() == 2:
            mask = mask[:, None, None, :].to(torch.bool)
        elif mask.dim() == 4 and mask.dtype != torch.bool:
            mask = mask.eq(0)
        if mask.shape[-1] > k_t.shape[-2]:
            mask = mask[..., -k_t.shape[-2]:]
    groups = q_t.shape[1] // k_t.shape[1]
    if groups != 1:
        k_t = k_t.repeat_interleave(groups, dim=1)
        v_t = v_t.repeat_interleave(groups, dim=1)
    if batch_one:
        scores = torch.matmul(q_t * (q_t.shape[-1] ** -0.5), k_t.transpose(-2, -1))
        probs = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_t.dtype)
        out = torch.matmul(probs, v_t)
    else:
        out = F.scaled_dot_product_attention(
            q_t,
            k_t,
            v_t,
            attn_mask=mask,
            dropout_p=0.0,
            is_causal=(mask is None and q_t.shape[-2] == k_t.shape[-2]),
        )
    return out.transpose(1, 2).contiguous()


cases = [
    {"name": "batched_gqa_decode", "batch": 4, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17},
    {"name": "batched_equal_heads_causal", "batch": 4, "q_heads": 8, "kv_heads": 8, "q_len": 3, "kv_len": 3},
    {"name": "batch_one_explicit", "batch": 1, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17},
    {"name": "sliding_window", "batch": 4, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17, "sliding_window": 8},
    {"name": "batched_masked_decode", "batch": 4, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17, "masked": True},
]
for case in cases:
    q = torch.randn(case["batch"], case["q_len"], case["q_heads"], 16, device=device, dtype=dtype)
    k = torch.randn(case["batch"], case["kv_len"], case["kv_heads"], 16, device=device, dtype=dtype)
    v = torch.randn(case["batch"], case["kv_len"], case["kv_heads"], 16, device=device, dtype=dtype)
    mask = None
    if case.get("masked"):
        mask = torch.ones(case["batch"], case["kv_len"], device=device, dtype=torch.long)
        mask[:, :2] = 0
    window = case.get("sliding_window")
    actual = fn(q, k, v, attention_mask=mask, sliding_window=window)
    expected = explicit_reference(
        q,
        k,
        v,
        mask=mask,
        sliding_window=window,
        batch_one=(case["batch"] == 1),
    )
    torch.cuda.synchronize()
    if actual.shape != expected.shape or not torch.isfinite(actual).all():
        raise RuntimeError(
            f"PROGRAM063_QWEN3_BAD_OUTPUT case={case['name']} actual={actual.shape} expected={expected.shape}"
        )
    max_abs_error = float((actual.float() - expected.float()).abs().max().cpu())
    if max_abs_error > 0.03:
        raise RuntimeError(
            f"PROGRAM063_QWEN3_PARITY_FAILED case={case['name']} max_abs_error={max_abs_error}"
        )
    reports.append({
        "name": case["name"],
        "batch": case["batch"],
        "q_heads": case["q_heads"],
        "kv_heads": case["kv_heads"],
        "q_len": case["q_len"],
        "kv_len": case["kv_len"],
        "sliding_window": window,
        "masked": bool(case.get("masked")),
        "shape": list(actual.shape),
        "max_abs_error": max_abs_error,
    })
print("PROGRAM063_QWEN3_OFFICIAL_KV_PROBE_OK=" + json.dumps({
    "backend": backend,
    "torch": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "cases": reports,
}, sort_keys=True))
''',
        encoding="utf-8",
    )

def bootstrap_source(program_no: int) -> str:
    return f'''import os\nimport sys\nfrom pathlib import Path\nsource_root = Path(os.environ["PROGRAM{program_no:03d}_LEGACY_ROOT"]).resolve()\noverlay_root = Path(os.environ["PROGRAM{program_no:03d}_OVERLAY_ROOT"]).resolve()\nclean = []\nfor item in sys.path:\n    if not item:\n        clean.append(item)\n        continue\n    try:\n        resolved = Path(item).resolve()\n    except OSError:\n        continue\n    if resolved == source_root or source_root in resolved.parents or resolved == overlay_root:\n        continue\n    clean.append(item)\nsys.path[:] = [str(overlay_root)] + clean\n'''


def write_import_probe(path: Path, program_no: int) -> None:
    path.write_text(
        bootstrap_source(program_no)
        + f'''
import importlib
import json
from pathlib import Path

# Match production import order: native Torch/Triton first, then Unsloth before
# Transformers/PEFT/TRL so Unsloth can apply its patches deterministically.
modules = {{}}
for name in ["torch", "triton"]:
    module = importlib.import_module(name)
    modules[name] = {{
        "version": getattr(module, "__version__", "unknown"),
        "file": str(Path(module.__file__).resolve()) if getattr(module, "__file__", None) else None,
    }}
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import unsloth
for name in ["transformers", "datasets", "peft", "accelerate", "bitsandbytes", "trl"]:
    module = importlib.import_module(name)
    modules[name] = {{
        "version": getattr(module, "__version__", "unknown"),
        "file": str(Path(module.__file__).resolve()) if getattr(module, "__file__", None) else None,
    }}
import torch
source_root = Path(os.environ["PROGRAM{program_no:03d}_LEGACY_ROOT"]).resolve()
overlay_root = Path(os.environ["PROGRAM{program_no:03d}_OVERLAY_ROOT"]).resolve()
for anchor in ["torch", "triton"]:
    anchor_path = Path(modules[anchor]["file"]).resolve()
    if anchor_path == source_root or source_root in anchor_path.parents or anchor_path == overlay_root or overlay_root in anchor_path.parents:
        raise RuntimeError(f"{{anchor}} was imported from a legacy overlay: {{anchor_path}}")
visible = torch.cuda.device_count()
if visible < 4:
    raise RuntimeError(f"Expected four visible GPUs, found {{visible}}")
optional = {{}}
for name in ["flash_attn", "xformers", "cut_cross_entropy", "hf_transfer", "msgspec", "sentencepiece"]:
    try:
        module = importlib.import_module(name)
        optional[name] = {{"ok": True, "version": getattr(module, "__version__", "unknown"), "file": str(Path(module.__file__).resolve()) if getattr(module, "__file__", None) else None}}
    except Exception as exc:
        optional[name] = {{"ok": False, "error": f"{{type(exc).__name__}}: {{exc}}"}}
print("PROGRAM{program_no:03d}_IMPORT_PROBE=" + json.dumps({{
    "python": str(Path(sys.executable).resolve()),
    "python_version": sys.version.split()[0],
    "modules": modules,
    "unsloth_version": getattr(unsloth, "__version__", "unknown"),
    "unsloth_file": str(Path(unsloth.__file__).resolve()),
    "optional": optional,
    "cuda_version": torch.version.cuda,
    "device_count": visible,
    "device_names": [torch.cuda.get_device_name(i) for i in range(visible)],
    "required_symbols": {{
        "FastLanguageModel": FastLanguageModel is not None,
        "UnslothTrainer": UnslothTrainer is not None,
        "UnslothTrainingArguments": UnslothTrainingArguments is not None,
    }},
}}, sort_keys=True))
''',
        encoding="utf-8",
    )


def parse_missing_module(stderr: str) -> str | None:
    matches = re.findall(r"ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", stderr)
    return matches[-1].split(".", 1)[0] if matches else None


def build_and_probe_profile(
    profile_name: str,
    initial_distributions: list[str],
    source_root: Path,
    by_name: dict[str, Path],
    module_to_dist: dict[str, str],
    base_env: dict[str, str],
    probe_path: Path,
) -> tuple[dict | None, dict]:
    overlay_root = PROFILE_ROOT / profile_name
    if overlay_root.exists():
        shutil.rmtree(overlay_root)
    overlay_root.mkdir(parents=True, exist_ok=True)
    copied_distributions: list[str] = []
    copied_entries: list[str] = []
    errors: list[dict] = []

    for dist in initial_distributions:
        try:
            copied_entries.extend(copy_distribution(source_root, overlay_root, dist, by_name))
            copied_distributions.append(canonical_name(dist))
        except FileNotFoundError:
            if canonical_name(dist) in {"unsloth", "unsloth-zoo", "transformers", "peft", "accelerate", "datasets", "trl"}:
                raise
    patches = patch_legacy_transformers(overlay_root)

    for attempt in range(1, 25):
        env = base_env.copy()
        env.update({
            "PROGRAM063_LEGACY_ROOT": str(source_root),
            "PROGRAM063_OVERLAY_ROOT": str(overlay_root),
            "PYTHONPATH": sanitize_pythonpath(base_env.get("PYTHONPATH", ""), source_root, overlay_root),
        })
        result = subprocess.run(
            [sys.executable, str(probe_path)],
            env=env,
            cwd=str(WORK_ROOT),
            text=True,
            capture_output=True,
        )
        if result.stdout:
            print(result.stdout, end="")
        if result.returncode == 0:
            marker = next((line for line in result.stdout.splitlines() if line.startswith("PROGRAM063_IMPORT_PROBE=")), None)
            if marker is None:
                raise RuntimeError(f"Profile {profile_name} succeeded without an import marker")
            info = json.loads(marker.split("=", 1)[1])
            info.update({
                "profile": profile_name,
                "overlay_root": str(overlay_root),
                "runtime_pythonpath": env["PYTHONPATH"],
                "copied_distributions": copied_distributions,
                "copied_entries": copied_entries,
                "patches": patches,
            })
            return info, {"profile": profile_name, "success": True, "attempts": attempt}

        missing = parse_missing_module(result.stderr)
        errors.append({
            "attempt": attempt,
            "returncode": result.returncode,
            "missing": missing,
            "stderr_tail": " | ".join(result.stderr.strip().splitlines()[-10:]),
        })
        if not missing or missing in NATIVE_BINARY_ANCHORS:
            break
        dist = module_to_dist.get(missing)
        if not dist or dist in copied_distributions:
            break
        try:
            copied_entries.extend(copy_distribution(source_root, overlay_root, dist, by_name))
            copied_distributions.append(dist)
            patches.extend(patch_legacy_transformers(overlay_root))
            print(f"PROGRAM063_PROFILE_RETRY profile={profile_name} missing={missing} copied={dist}")
        except Exception as exc:
            errors.append({"attempt": attempt, "copy_error": f"{type(exc).__name__}: {exc}"})
            break

    return None, {
        "profile": profile_name,
        "success": False,
        "copied_distributions": copied_distributions,
        "copied_entries": copied_entries,
        "patches": patches,
        "errors": errors,
    }


rerun_mode = env_flag("KAGGLE_IS_COMPETITION_RERUN")
challenge_names = (
    ["arc-agi_test_challenges.json", "arc_agi_test_challenges.json"]
    if rerun_mode
    else ["arc-agi_evaluation_challenges.json", "arc_agi_evaluation_challenges.json"]
)
challenge_path = resolve_named_file(challenge_names)
solution_path = None if rerun_mode else resolve_named_file(
    ["arc-agi_evaluation_solutions.json", "arc_agi_evaluation_solutions.json"]
)
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Model directory is missing: {MODEL_PATH}")

ptxas_path = resolve_ptxas()
ptxas_version = subprocess.check_output([str(ptxas_path), "--version"], text=True, stderr=subprocess.STDOUT).splitlines()[0]
legacy_roots = discover_legacy_roots()
runtime_selection_attempts: list[dict] = []
selected_runtime_family: str | None = None
legacy_root: Path | None = None
by_name: dict[str, Path] | None = None
module_to_dist: dict[str, str] | None = None
available_versions: dict[str, list[str]] | None = None
pinned_source_report: dict | None = None

for family_name, family_lock in RUNTIME_FAMILIES:
    for candidate_root in legacy_roots:
        try:
            candidate_by_name, candidate_module_map, candidate_versions = distribution_index(
                candidate_root, family_lock, family_name
            )
            PINNED_RUNTIME_DISTRIBUTIONS = family_lock
            candidate_report = verify_pinned_runtime(candidate_root, candidate_by_name)
        except Exception as exc:
            runtime_selection_attempts.append({
                "runtime_family": family_name,
                "source_root": str(candidate_root),
                "success": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
            continue
        selected_runtime_family = family_name
        legacy_root = candidate_root
        by_name = candidate_by_name
        module_to_dist = candidate_module_map
        available_versions = candidate_versions
        pinned_source_report = candidate_report
        runtime_selection_attempts.append({
            "runtime_family": family_name,
            "source_root": str(candidate_root),
            "success": True,
            "versions": {
                name: lock["version"] for name, lock in sorted(family_lock.items())
            },
        })
        break
    if selected_runtime_family is not None:
        break

if (
    selected_runtime_family is None
    or legacy_root is None
    or by_name is None
    or module_to_dist is None
    or available_versions is None
    or pinned_source_report is None
):
    raise RuntimeError(
        "PROGRAM063_NO_VERIFIED_RUNTIME_FAMILY="
        + json.dumps({
            "roots": [str(root) for root in legacy_roots],
            "attempts": runtime_selection_attempts,
        }, sort_keys=True)
    )

print("PROGRAM063_RUNTIME_FAMILY_SELECTED=" + json.dumps({
    "runtime_family": selected_runtime_family,
    "source_root": str(legacy_root),
    "versions": {
        name: lock["version"] for name, lock in sorted(PINNED_RUNTIME_DISTRIBUTIONS.items())
    },
    "root_count": len(legacy_roots),
    "attempt_count": len(runtime_selection_attempts),
}, sort_keys=True))
print("PROGRAM063_PINNED_RUNTIME_SOURCE_OK=" + json.dumps({
    "runtime_family": selected_runtime_family,
    "source_root": str(legacy_root),
    "available_versions": {
        name: available_versions.get(name, [])
        for name in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    },
    "locked": pinned_source_report,
}, sort_keys=True))
PROFILE_ROOT.mkdir(parents=True, exist_ok=True)

base_env = os.environ.copy()
base_env.update({
    "TRITON_PTXAS_PATH": str(ptxas_path),
    "PYTHONHASHSEED": PYTHON_HASH_SEED,
    "PROGRAM063_BASE_SEED": str(BASE_SEED),
    "PROGRAM063_SEED_POLICY": "rng_only_no_backend_change",
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})

probe_path = WORK_ROOT / "program063_import_probe.py"
write_import_probe(probe_path, PROGRAM_NO)

profiles = [
    # Program 063 intentionally has no native-core fallback. A fallback could
    # silently reintroduce the newer Accelerate/PEFT/bitsandbytes combination.
    (
        "legacy_python_stack_pinned",
        [
            "unsloth", "unsloth-zoo", "transformers", "peft", "accelerate",
            "datasets", "trl", "huggingface-hub", "bitsandbytes",
        ],
    ),
]

selected = None
profile_reports = []
for profile_name, initial in profiles:
    try:
        info, report = build_and_probe_profile(
            profile_name, initial, legacy_root, by_name, module_to_dist, base_env, probe_path
        )
    except Exception as exc:
        info = None
        report = {"profile": profile_name, "success": False, "build_error": f"{type(exc).__name__}: {exc}"}
    profile_reports.append(report)
    print("PROGRAM063_PROFILE_REPORT=" + json.dumps(report, sort_keys=True))
    if info is not None:
        selected = info
        break

if selected is None:
    raise RuntimeError(
        "PROGRAM063_RUNTIME_PROFILE_FAILED: the exact pinned legacy Python-stack profile could not "
        "import the complete Unsloth runtime. No native-core fallback is permitted. "
        "Aborting before every GPU probe and model loading. Reports="
        + json.dumps(profile_reports, sort_keys=True)
    )

imported_pinned_versions = {}
for module_name, lock in sorted(PINNED_RUNTIME_DISTRIBUTIONS.items()):
    actual = selected.get("modules", {}).get(module_name, {}).get("version")
    imported_pinned_versions[module_name] = actual
    if actual != lock["version"]:
        raise RuntimeError(
            "PROGRAM063_PINNED_IMPORT_VERSION_MISMATCH="
            + json.dumps({
                "module": module_name,
                "expected": lock["version"],
                "actual": actual,
                "module_info": selected.get("modules", {}).get(module_name),
            }, sort_keys=True)
        )

overlay_root_path = Path(selected["overlay_root"])
overlay_by_name, _, overlay_available_versions = distribution_index(
    overlay_root_path, PINNED_RUNTIME_DISTRIBUTIONS, selected_runtime_family
)
pinned_overlay_report = verify_pinned_runtime(overlay_root_path, overlay_by_name)
for module_name in sorted(PINNED_RUNTIME_DISTRIBUTIONS):
    source_fingerprint = pinned_source_report[module_name]["record_tree_sha256"]
    overlay_fingerprint = pinned_overlay_report[module_name]["record_tree_sha256"]
    if source_fingerprint != overlay_fingerprint:
        raise RuntimeError(
            "PROGRAM063_PINNED_OVERLAY_FINGERPRINT_MISMATCH="
            + json.dumps({
                "module": module_name,
                "source": source_fingerprint,
                "overlay": overlay_fingerprint,
            }, sort_keys=True)
        )
print("PROGRAM063_PINNED_RUNTIME_OVERLAY_OK=" + json.dumps({
    "imported_versions": imported_pinned_versions,
    "overlay_available_versions": {
        name: overlay_available_versions.get(name, [])
        for name in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    },
    "locked": pinned_overlay_report,
}, sort_keys=True))

qwen3_patch_report = install_qwen3_official_kv_cache_patch(Path(selected["overlay_root"]))
print("PROGRAM063_QWEN3_OFFICIAL_KV_PATCH_OK=" + json.dumps(qwen3_patch_report, sort_keys=True))

selected_env = base_env.copy()
selected_env.update({
    "PROGRAM063_LEGACY_ROOT": str(legacy_root),
    "PROGRAM063_OVERLAY_ROOT": selected["overlay_root"],
    "PYTHONPATH": selected["runtime_pythonpath"],
})

qwen3_probe_path = WORK_ROOT / "program063_qwen3_official_kv_probe.py"
write_qwen3_official_kv_probe(qwen3_probe_path, PROGRAM_NO)
qwen3_probe_env = selected_env.copy()
qwen3_probe_env.update({"CUDA_VISIBLE_DEVICES": "0"})
qwen3_probe_result = subprocess.run(
    [sys.executable, str(qwen3_probe_path)],
    env=qwen3_probe_env,
    cwd=str(WORK_ROOT),
    text=True,
    capture_output=True,
    timeout=180,
)
if qwen3_probe_result.stdout:
    print(qwen3_probe_result.stdout, end="")
if qwen3_probe_result.returncode != 0:
    if qwen3_probe_result.stderr:
        print(qwen3_probe_result.stderr, file=sys.stderr, end="")
    raise RuntimeError(
        f"PROGRAM063_QWEN3_OFFICIAL_KV_PROBE_FAILED exit_code={qwen3_probe_result.returncode}"
    )
qwen3_probe_marker = next(
    (
        line
        for line in qwen3_probe_result.stdout.splitlines()
        if line.startswith("PROGRAM063_QWEN3_OFFICIAL_KV_PROBE_OK=")
    ),
    None,
)
if qwen3_probe_marker is None:
    raise RuntimeError("PROGRAM063_QWEN3_OFFICIAL_KV_PROBE_MARKER_MISSING")
qwen3_probe_report = json.loads(qwen3_probe_marker.split("=", 1)[1])

triton_probe_path = WORK_ROOT / "program063_triton_probe.py"
triton_probe_path.write_text(
    bootstrap_source(PROGRAM_NO)
    + r'''
import torch
import triton
import triton.language as tl

@triton.jit
def copy_kernel(x_ptr, y_ptr, n_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    values = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, values, mask=mask)

x = torch.arange(16, device="cuda:0", dtype=torch.float32)
y = torch.empty_like(x)
copy_kernel[(1,)](x, y, n_elements=16, BLOCK_SIZE=16)
torch.cuda.synchronize()
if not torch.equal(x, y):
    raise RuntimeError("Triton probe produced an incorrect result")
print("PROGRAM063_TRITON_PROBE_OK")
''',
    encoding="utf-8",
)
triton_result = subprocess.run(
    [sys.executable, str(triton_probe_path)],
    env=selected_env,
    cwd=str(WORK_ROOT),
    text=True,
    capture_output=True,
)
if triton_result.stdout:
    print(triton_result.stdout, end="")
if triton_result.returncode != 0:
    if triton_result.stderr:
        print(triton_result.stderr, file=sys.stderr, end="")
    raise RuntimeError(f"PROGRAM063_TRITON_PROBE_FAILED exit_code={triton_result.returncode}")

run_plan = {
    "program_no": PROGRAM_NO,
    "rerun_mode": rerun_mode,
    "challenge_path": str(challenge_path),
    "solution_path": str(solution_path) if solution_path else None,
    "model_path": str(MODEL_PATH),
    "ptxas_path": str(ptxas_path),
    "ptxas_version": ptxas_version,
    "legacy_root": str(legacy_root),
    "overlay_root": selected["overlay_root"],
    "runtime_pythonpath": selected["runtime_pythonpath"],
    "python_executable": str(Path(sys.executable).resolve()),
    "python_version": selected["python_version"],
    "runtime_profile": selected["profile"],
    "runtime_family": selected_runtime_family,
    "runtime_selection_attempts": runtime_selection_attempts,
    "runtime_info": selected,
    "pinned_runtime_versions": {
        name: lock["version"] for name, lock in sorted(PINNED_RUNTIME_DISTRIBUTIONS.items())
    },
    "pinned_runtime_source": pinned_source_report,
    "pinned_runtime_overlay": pinned_overlay_report,
    "pinned_runtime_imported_versions": imported_pinned_versions,
    "pinned_runtime_available_versions": {
        name: available_versions.get(name, [])
        for name in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    },
    "qwen3_official_kv_patch": qwen3_patch_report,
    "qwen3_official_kv_probe": qwen3_probe_report,
    "seed_contract": SEED_CONTRACT,
    "global_end_time": global_end_time,
    "notebook_start_time": NOTEBOOK_START_TIME,
}
Path("program063_run_plan.json").write_text(json.dumps(run_plan, indent=2, sort_keys=True), encoding="utf-8")

print(
    "PROGRAM063_PREFLIGHT_OK "
    f"profile={selected['profile']} python={selected['python_version']} "
    f"torch={selected['modules']['torch']['version']} triton={selected['modules']['triton']['version']} "
    f"transformers={selected['modules']['transformers']['version']} unsloth={selected['unsloth_version']} "
    f"gpus={selected['device_count']} "
    f"runtime_family={selected_runtime_family} "
    f"runtime_lock=accelerate{PINNED_RUNTIME_DISTRIBUTIONS['accelerate']['version']}_"
    f"peft{PINNED_RUNTIME_DISTRIBUTIONS['peft']['version']}_"
    f"bnb{PINNED_RUNTIME_DISTRIBUTIONS['bitsandbytes']['version']} "
    f"seed_policy=rng_only base_seed={BASE_SEED} python_hash_seed={PYTHON_HASH_SEED} "
    f"qwen3_attention={qwen3_probe_report['backend']} "
    f"challenge={challenge_path}"
)


In [ ]:
# Remove TensorFlow only after the complete offline runtime profile and Triton kernel have passed.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow"], check=False)


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time, test_path):

    if not os.path.isfile(test_path):
        raise FileNotFoundError(f"Resolved ARC challenge file is missing: {test_path}")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import hashlib
import json
import os
import pickle
import random
import shutil
import sys
import time
from pathlib import Path


def sanitize_runtime_path():
    source_raw = os.environ.get("PROGRAM063_LEGACY_ROOT")
    overlay_raw = os.environ.get("PROGRAM063_OVERLAY_ROOT")
    if not source_raw or not overlay_raw:
        raise RuntimeError("Program No. 063 runtime path variables are missing")
    source_root = Path(source_raw).resolve()
    overlay_root = Path(overlay_raw).resolve()
    clean = []
    for item in sys.path:
        if not item:
            clean.append(item)
            continue
        try:
            resolved = Path(item).resolve()
        except OSError:
            continue
        if resolved == source_root or source_root in resolved.parents:
            continue
        if resolved == overlay_root:
            continue
        clean.append(item)
    sys.path[:] = [str(overlay_root)] + clean


sanitize_runtime_path()

import numpy as np
import torch
import argparse
import torch.multiprocessing as mp


STARTER_LOCK = Path("/kaggle/working/program063_starter.lock")
SEED_PROBE_TEXT = "arc-program-seed-probe-v1"


def env_flag(name):
    value = os.getenv(name)
    if value is None:
        return False
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def seed_contract():
    raw_seed = os.environ.get("PROGRAM063_BASE_SEED")
    if raw_seed is None:
        raise RuntimeError("PROGRAM063_BASE_SEED is missing")
    try:
        base_seed = int(raw_seed)
    except ValueError as exc:
        raise RuntimeError(f"PROGRAM063_BASE_SEED is invalid: {raw_seed!r}") from exc
    python_hash_seed = os.environ.get("PYTHONHASHSEED")
    if python_hash_seed != "0":
        raise RuntimeError(
            f"PROGRAM063_PYTHONHASHSEED_INVALID expected='0' actual={python_hash_seed!r}"
        )
    policy = os.environ.get("PROGRAM063_SEED_POLICY")
    if policy != "rng_only_no_backend_change":
        raise RuntimeError(f"PROGRAM063_SEED_POLICY_INVALID={policy!r}")
    return base_seed, python_hash_seed, policy


def rng_state_report(stage, rank, task_id=None):
    base_seed, python_hash_seed, policy = seed_contract()
    random.seed(base_seed)
    np.random.seed(base_seed)
    torch.manual_seed(base_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(base_seed)

    python_state = pickle.dumps(random.getstate(), protocol=4)
    numpy_state = pickle.dumps(np.random.get_state(), protocol=4)
    torch_cpu_state = torch.get_rng_state().cpu().numpy().tobytes()
    cuda_state_hashes = []
    if torch.cuda.is_available():
        for state in torch.cuda.get_rng_state_all():
            cuda_state_hashes.append(sha256_bytes(state.cpu().numpy().tobytes()))

    return {
        "stage": stage,
        "rank": rank,
        "task_id": task_id,
        "base_seed": base_seed,
        "python_hash_seed": python_hash_seed,
        "seed_policy": policy,
        "hash_probe": hash(SEED_PROBE_TEXT),
        "python_rng_sha256": sha256_bytes(python_state),
        "numpy_rng_sha256": sha256_bytes(numpy_state),
        "torch_cpu_rng_sha256": sha256_bytes(torch_cpu_state),
        "torch_cuda_rng_sha256": cuda_state_hashes,
        "torch_initial_seed": int(torch.initial_seed()),
        "deterministic_algorithms_enabled": bool(torch.are_deterministic_algorithms_enabled()),
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    }


class SeededQueueProxy:
    def __init__(self, queue, rank):
        self._queue = queue
        self._rank = rank

    def get(self, *args, **kwargs):
        item = self._queue.get(*args, **kwargs)
        if item is not None:
            report = rng_state_report("task_dequeue", self._rank, str(item))
            print("PROGRAM063_TASK_SEED_READY=" + json.dumps(report, sort_keys=True), flush=True)
        return item


def acquire_single_starter_lock():
    try:
        fd = os.open(STARTER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)
    except FileExistsError as exc:
        owner = STARTER_LOCK.read_text(encoding="utf-8", errors="replace") if STARTER_LOCK.exists() else "unknown"
        raise RuntimeError(f"PROGRAM063_DUPLICATE_STARTER_BLOCKED existing={owner}") from exc
    try:
        os.write(fd, f"pid={os.getpid()} started={time.time():.6f}\n".encode("utf-8"))
    finally:
        os.close(fd)
    print(f"PROGRAM063_STARTER_LOCK_ACQUIRED pid={os.getpid()} path={STARTER_LOCK}", flush=True)


def release_single_starter_lock():
    try:
        STARTER_LOCK.unlink()
    except FileNotFoundError:
        pass


def local_worker(rank, queue, end_time, test_path):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")
    worker_seed_report = rng_state_report("worker_init", rank)
    print("PROGRAM063_WORKER_SEED_READY=" + json.dumps(worker_seed_report, sort_keys=True), flush=True)

    if rank > 0:
        previous_marker = Path(f"/kaggle/worker{rank - 1}")
        while not previous_marker.exists():
            if time.time() >= end_time:
                raise TimeoutError(
                    f"Rank {rank} reached the global deadline while waiting for {previous_marker}"
                )
            time.sleep(2)

    # Import the byte-identical Program 024 solver only after this process has
    # received its fixed Python/NumPy/Torch seed state.
    from arc_solver import worker

    Path(f"/kaggle/worker{rank}").write_text(
        json.dumps({"rank": rank, "pid": os.getpid(), "gpu": rank}, sort_keys=True),
        encoding="utf-8",
    )
    print(f"[Rank {rank}] start! pid={os.getpid()} gpu={rank}", flush=True)
    worker(rank, SeededQueueProxy(queue, rank), end_time, test_path)
    print(f"[Rank {rank}] done! pid={os.getpid()} gpu={rank}", flush=True)


def clean_runtime_state():
    output_dir = Path("/kaggle/inference_outputs")
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for rank in range(4):
        marker = Path(f"/kaggle/worker{rank}")
        if marker.exists():
            marker.unlink()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, required=True)
    parser.add_argument("--challenge-path", type=str, required=True)
    args = parser.parse_args()
    base_seed, python_hash_seed, seed_policy = seed_contract()
    acquire_single_starter_lock()
    try:
        test_path = str(Path(args.challenge_path).resolve())
        if not os.path.isfile(test_path):
            raise FileNotFoundError(f"Challenge file not found: {test_path}")
        with open(test_path, "r", encoding="utf-8") as handle:
            data = json.load(handle)
        if not isinstance(data, dict) or not data:
            raise ValueError("Challenge JSON must be a non-empty object")
        visible_gpus = torch.cuda.device_count()
        if visible_gpus < 4:
            raise RuntimeError(
                f"Program No. 063 requires four visible GPUs; detected {visible_gpus}. "
                "Aborting before model loading."
            )
        torch_file = Path(torch.__file__).resolve()
        source_root = Path(os.environ["PROGRAM063_LEGACY_ROOT"]).resolve()
        if torch_file == source_root or source_root in torch_file.parents:
            raise RuntimeError(f"Program No. 063 loaded legacy Torch unexpectedly: {torch_file}")
        clean_runtime_state()
        rerun_mode = env_flag("KAGGLE_IS_COMPETITION_RERUN")
        selected_keys = []
        for key in sorted(data.keys()):
            if not rerun_mode and key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
            selected_keys.append(key)
        if not selected_keys:
            raise RuntimeError("No ARC tasks were selected")
        manager = mp.Manager()
        queue = manager.Queue()
        for key in selected_keys:
            queue.put(key)
        for _ in range(4):
            queue.put(None)
        print(
            f"PROGRAM063_STARTER_READY parent_pid={os.getpid()} rerun_mode={rerun_mode} "
            f"tasks={len(selected_keys)} gpus={visible_gpus} torch={torch_file} "
            f"base_seed={base_seed} python_hash_seed={python_hash_seed} seed_policy={seed_policy}",
            flush=True,
        )
        mp.spawn(local_worker, args=(queue, args.end_time, test_path), nprocs=4)
        print(
            f"PROGRAM063_STARTER_COMPLETE parent_pid={os.getpid()} workers=4 tasks={len(selected_keys)}",
            flush=True,
        )
    finally:
        release_single_starter_lock()


In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import py_compile
import re
import subprocess
import sys
import time
from pathlib import Path


def normalize_python_source(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n").rstrip("\n")


def normalized_source_sha256(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    return hashlib.sha256(normalize_python_source(text).encode("utf-8")).hexdigest()


EXPECTED_PROGRAM024_HASHES = {
    "arc_loader.py": "d01cd56167e534ae156706ef62ab11662fc7b403964614a7a55131940bde970c",
    "arc_decoder.py": "965cfd910777d9bbec9681c6c15c5e2ea3924569734248046f5e58f16cb28222",
    "arc_solver.py": "f9011b4d4549fb688488d67ba7c2dc43184824436d34e1a4c2eae06f69daab73",
}
actual_hashes = {
    name: normalized_source_sha256(name)
    for name in EXPECTED_PROGRAM024_HASHES
}
if actual_hashes != EXPECTED_PROGRAM024_HASHES:
    raise RuntimeError(
        "PROGRAM063_PROGRAM024_SOURCE_MISMATCH="
        + json.dumps({"expected": EXPECTED_PROGRAM024_HASHES, "actual": actual_hashes}, sort_keys=True)
    )
print("PROGRAM063_EXACT_PROGRAM024_SOURCE_HASH_OK=" + json.dumps(actual_hashes, sort_keys=True))

for script in ["arc_loader.py", "arc_decoder.py", "arc_solver.py", "starter.py"]:
    py_compile.compile(script, doraise=True)

solver_source = Path("arc_solver.py").read_text(encoding="utf-8")
decoder_source = Path("arc_decoder.py").read_text(encoding="utf-8")
starter_source = Path("starter.py").read_text(encoding="utf-8")
solver_tree = ast.parse(solver_source, filename="arc_solver.py")
decoder_tree = ast.parse(decoder_source, filename="arc_decoder.py")
ast.parse(starter_source, filename="starter.py")

peft_params = None
for node in ast.walk(solver_tree):
    if isinstance(node, ast.Assign) and any(
        isinstance(target, ast.Name) and target.id == "peft_params"
        for target in node.targets
    ):
        if isinstance(node.value, ast.Call) and isinstance(node.value.func, ast.Name) and node.value.func.id == "dict":
            peft_params = {kw.arg: ast.literal_eval(kw.value) for kw in node.value.keywords}
        else:
            peft_params = ast.literal_eval(node.value)
        break
if peft_params is None:
    raise RuntimeError("PROGRAM063_PEFT_PARAMS_MISSING")
for key, expected in {"r": 256, "lora_alpha": 32, "use_rslora": True}.items():
    if peft_params.get(key) != expected:
        raise RuntimeError(
            f"PROGRAM063_PEFT_PARAM_MISMATCH key={key} actual={peft_params.get(key)!r} expected={expected!r}"
        )

required_patterns = {
    "original training augmentation": r"train_ds\s*=\s*puzzle_ds\.augment\(n=16,\s*shfl_keys=True,\s*seed=1\)",
    "original evaluation augmentation": r"eval_ds\s*=\s*puzzle_ds_multi\.augment\(n=2,\s*seed=2\)",
    "original scoring augmentation": r"aug_dataset\s*=\s*aug_dataset\.augment\(seed=hash\(bk\)\s*%\s*1024\*\*2\)",
    "original DFS window": r"time\.time\(\)\s*-\s*start_time\s*<\s*540\b",
    "original task timeout": r"spend_time\s*>\s*1200\b",
    "blocking queue": r"\bkey\s*=\s*queue\.get\(\)",
    "queue sentinel": r"if\s+key\s+is\s+None\s*:",
}
for label, pattern in required_patterns.items():
    if re.search(pattern, solver_source) is None:
        raise RuntimeError(f"PROGRAM063_BASELINE_INVARIANT_MISSING={label}")

# KGMon is implemented in arc_decoder.py, not arc_solver.py. Verify it structurally
# instead of using a case-sensitive text search in the wrong file.
decoder_functions = {
    node.name: node
    for node in decoder_tree.body
    if isinstance(node, ast.FunctionDef)
}
if "score_kgmon" not in decoder_functions:
    raise RuntimeError("PROGRAM063_KGMON_SCORE_FUNCTION_MISSING")
selection_list_ok = False
run_selection_default_ok = False
for node in decoder_tree.body:
    if isinstance(node, ast.Assign) and any(
        isinstance(target, ast.Name) and target.id == "selection_algorithms"
        for target in node.targets
    ):
        if isinstance(node.value, (ast.List, ast.Tuple)):
            selection_list_ok = any(
                isinstance(item, ast.Name) and item.id == "score_kgmon"
                for item in node.value.elts
            )
    if isinstance(node, ast.ClassDef) and node.name == "ArcDecoder":
        for method in node.body:
            if isinstance(method, ast.FunctionDef) and method.name == "run_selection_algo":
                defaults = method.args.defaults
                run_selection_default_ok = bool(
                    defaults
                    and isinstance(defaults[-1], ast.Name)
                    and defaults[-1].id == "score_kgmon"
                )
if not selection_list_ok:
    raise RuntimeError("PROGRAM063_KGMON_SELECTION_LIST_MISSING")
if not run_selection_default_ok:
    raise RuntimeError("PROGRAM063_KGMON_DEFAULT_SELECTION_MISMATCH")
print("PROGRAM063_KGMON_AST_OK")

for forbidden in [
    "build_factorized_symmetry_product",
    "build_color_descriptors",
    "stable_task_seed",
    "SYMMETRY_PLAN",
    "score_dual_ranker",
    "score_shape_mismatch_only_rescue",
    "replace_structurally_redundant_second",
    "borrowed_seconds",
    "deadline_blocked_tasks",
    "ARC_DFS_FORWARD_BUDGET",
    "while not queue.empty()",
]:
    if forbidden in solver_source:
        raise RuntimeError(f"PROGRAM063_FORBIDDEN_EXPERIMENTAL_CODE={forbidden}")

for marker in [
    "clean_runtime_state()",
    "marker.unlink()",
    "shutil.rmtree(output_dir)",
    "acquire_single_starter_lock()",
    "PROGRAM063_STARTER_COMPLETE",
    "PROGRAM063_WORKER_SEED_READY",
    "PROGRAM063_TASK_SEED_READY",
    "SeededQueueProxy",
    "rng_state_report",
]:
    if marker not in starter_source:
        raise RuntimeError(f"PROGRAM063_STARTER_INVARIANT_MISSING={marker}")

run_plan = json.loads(Path("program063_run_plan.json").read_text(encoding="utf-8"))
if run_plan.get("program_no") != PROGRAM_NO:
    raise RuntimeError(f"PROGRAM063_RUN_PLAN_ID_MISMATCH={run_plan.get('program_no')}")
expected_seed_contract = {
    "base_seed": 42,
    "python_hash_seed": "0",
    "worker_init_reset": True,
    "task_dequeue_reset": True,
    "deterministic_algorithms_enabled": False,
    "backend_policy_changed": False,
}
if run_plan.get("seed_contract") != expected_seed_contract:
    raise RuntimeError(
        "PROGRAM063_SEED_CONTRACT_MISMATCH="
        + json.dumps({
            "expected": expected_seed_contract,
            "actual": run_plan.get("seed_contract"),
        }, sort_keys=True)
    )
legacy_root = Path(run_plan["legacy_root"]).resolve()
overlay_root = Path(run_plan["overlay_root"]).resolve()
python_executable = Path(run_plan["python_executable"]).resolve()
if not python_executable.is_file():
    raise RuntimeError(f"PROGRAM063_PYTHON_MISSING={python_executable}")
if not (overlay_root / "unsloth" / "__init__.py").is_file():
    raise RuntimeError(f"PROGRAM063_UNSLOTH_OVERLAY_MISSING={overlay_root}")
for compiled_name in ["torch", "triton", "numpy"]:
    if (overlay_root / compiled_name).exists():
        raise RuntimeError(f"PROGRAM063_COMPILED_PACKAGE_LEAK={compiled_name}")

allowed_runtime_families = {
    "program050_current": {
        "accelerate": "1.13.0",
        "peft": "0.18.1",
        "bitsandbytes": "0.49.2",
    },
    "program047_legacy": {
        "accelerate": "1.11.0",
        "peft": "0.17.1",
        "bitsandbytes": "0.48.2",
    },
}
runtime_family = run_plan.get("runtime_family")
if runtime_family not in allowed_runtime_families:
    raise RuntimeError(f"PROGRAM063_RUNTIME_FAMILY_INVALID={runtime_family!r}")
expected_pinned_versions = allowed_runtime_families[runtime_family]
selection_attempts = run_plan.get("runtime_selection_attempts")
if not isinstance(selection_attempts, list) or not any(
    item.get("success") is True and item.get("runtime_family") == runtime_family
    for item in selection_attempts
    if isinstance(item, dict)
):
    raise RuntimeError("PROGRAM063_RUNTIME_SELECTION_AUDIT_MISSING")
if run_plan.get("pinned_runtime_versions") != expected_pinned_versions:
    raise RuntimeError(
        "PROGRAM063_RUN_PLAN_RUNTIME_LOCK_MISMATCH="
        + json.dumps({
            "expected": expected_pinned_versions,
            "actual": run_plan.get("pinned_runtime_versions"),
        }, sort_keys=True)
    )
if run_plan.get("pinned_runtime_imported_versions") != expected_pinned_versions:
    raise RuntimeError(
        "PROGRAM063_IMPORTED_RUNTIME_LOCK_MISMATCH="
        + json.dumps(run_plan.get("pinned_runtime_imported_versions"), sort_keys=True)
    )
for report_key in ["pinned_runtime_source", "pinned_runtime_overlay"]:
    reports = run_plan.get(report_key, {})
    for module_name, expected_version in expected_pinned_versions.items():
        report = reports.get(module_name, {})
        if report.get("version") != expected_version or not report.get("record_tree_sha256"):
            raise RuntimeError(
                "PROGRAM063_RUNTIME_FINGERPRINT_MISSING="
                + json.dumps({"report_key": report_key, "module": module_name, "report": report}, sort_keys=True)
            )
        if report.get("record_tree_sha256") != run_plan["pinned_runtime_source"][module_name]["record_tree_sha256"]:
            raise RuntimeError(
                "PROGRAM063_RUNTIME_FINGERPRINT_MISMATCH="
                + json.dumps({"report_key": report_key, "module": module_name}, sort_keys=True)
            )
print("PROGRAM063_RUNTIME_LOCK_RECHECK_OK=" + json.dumps({
    "runtime_family": runtime_family,
    "versions": expected_pinned_versions,
    "tree_sha256": {
        module_name: run_plan["pinned_runtime_source"][module_name]["record_tree_sha256"]
        for module_name in sorted(expected_pinned_versions)
    },
}, sort_keys=True))

qwen3_probe = run_plan.get("qwen3_official_kv_probe", {})
if qwen3_probe.get("backend") != "official_style_kv_cache":
    raise RuntimeError(f"PROGRAM063_QWEN3_PROBE_INVALID={qwen3_probe}")

run_env = os.environ.copy()
run_env.update({
    "PROGRAM063_LEGACY_ROOT": str(legacy_root),
    "PROGRAM063_OVERLAY_ROOT": str(overlay_root),
    "PYTHONPATH": run_plan["runtime_pythonpath"],
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": run_plan["ptxas_path"],
    "OMP_NUM_THREADS": "12",
    "PYTHONHASHSEED": run_plan["seed_contract"]["python_hash_seed"],
    "PROGRAM063_BASE_SEED": str(run_plan["seed_contract"]["base_seed"]),
    "PROGRAM063_SEED_POLICY": "rng_only_no_backend_change",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})
subprocess.run(
    [str(python_executable), "-m", "py_compile", "arc_loader.py", "arc_decoder.py", "arc_solver.py", "starter.py"],
    env=run_env,
    cwd="/kaggle/working",
    check=True,
)

# A one-step production-runtime smoke test catches binary/API failures before the four workers load models.
bootstrap = '''import os\nimport sys\nfrom pathlib import Path\nlegacy_root = Path(os.environ["PROGRAM063_LEGACY_ROOT"]).resolve()\noverlay_root = Path(os.environ["PROGRAM063_OVERLAY_ROOT"]).resolve()\nclean = []\nfor item in sys.path:\n    if not item:\n        clean.append(item)\n        continue\n    try:\n        resolved = Path(item).resolve()\n    except OSError:\n        continue\n    if resolved == legacy_root or legacy_root in resolved.parents or resolved == overlay_root:\n        continue\n    clean.append(item)\nsys.path[:] = [str(overlay_root)] + clean\n'''
smoke_path = Path("/kaggle/working/program063_model_smoke.py")
smoke_path.write_text(
    bootstrap
    + r'''
import hashlib
import json
import math
import os
import pickle
import random

import numpy as np
import torch
import torch.nn.functional as F

BASE_SEED = int(os.environ["PROGRAM063_BASE_SEED"])
if os.environ.get("PYTHONHASHSEED") != "0":
    raise RuntimeError(f"PROGRAM063_SMOKE_PYTHONHASHSEED={os.environ.get('PYTHONHASHSEED')!r}")
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
print("PROGRAM063_SMOKE_SEED_READY=" + json.dumps({
    "base_seed": BASE_SEED,
    "python_hash_seed": os.environ.get("PYTHONHASHSEED"),
    "hash_probe": hash("arc-program-seed-probe-v1"),
    "python_rng_sha256": hashlib.sha256(pickle.dumps(random.getstate(), protocol=4)).hexdigest(),
    "numpy_rng_sha256": hashlib.sha256(pickle.dumps(np.random.get_state(), protocol=4)).hexdigest(),
    "torch_cpu_rng_sha256": hashlib.sha256(torch.get_rng_state().cpu().numpy().tobytes()).hexdigest(),
    "deterministic_algorithms_enabled": bool(torch.are_deterministic_algorithms_enabled()),
    "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
    "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
}, sort_keys=True))

from unsloth import FastLanguageModel, UnslothTrainingArguments
from peft import get_peft_model_state_dict
from datasets import Dataset
from arc_loader import QwenFormatter
from arc_solver import ARC_TOKENS, QwenDataCollatorForCompletionOnlyLM, UnslothFixedTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    full_finetuning=False,
    load_in_4bit=False,
    local_files_only=True,
    use_gradient_checkpointing=False,
    max_seq_length=8192,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=256,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
    use_rslora=True,
    loftq_config=None,
)
for _, parameter in model.named_parameters():
    if parameter.dtype == torch.float32:
        parameter.data = parameter.data.to(torch.bfloat16)
state = get_peft_model_state_dict(model, adapter_name="default")
if not state:
    raise RuntimeError("PROGRAM063_LORA_STATE_EMPTY")

formatter = QwenFormatter(tokenizer)
smoke_text = formatter.fmt_train([
    {"input": [[1, 0], [0, 1]], "output": [[1, 0], [0, 1]]},
    {"input": [[2, 0], [0, 2]], "output": [[2, 0], [0, 2]]},
])
collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)
args = UnslothTrainingArguments(
    output_dir="/kaggle/working/program063_smoke_output",
    per_device_eval_batch_size=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_steps=1,
    warmup_steps=0,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    learning_rate=5e-5,
    optim="adamw_torch",
    weight_decay=0.0,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    save_strategy="no",
    eval_strategy="no",
    logging_strategy="no",
    fp16=False,
    bf16=True,
    fsdp="",
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    gradient_checkpointing=False,
)
model = FastLanguageModel.for_training(model)
trainer = UnslothFixedTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=Dataset.from_list([{"text": smoke_text}]),
    dataset_text_field="text",
    max_seq_length=8192,
    args=args,
)
trainer.create_optimizer_and_scheduler(num_training_steps=1)
batch = next(iter(trainer.get_train_dataloader()))
batch = {
    key: (value.to(model.device) if torch.is_tensor(value) else value)
    for key, value in batch.items()
}
if "labels" not in batch or not torch.any(batch["labels"] != -100):
    raise RuntimeError("PROGRAM063_SMOKE_NO_SUPERVISED_LABELS")
loss = trainer.compute_loss(model, batch)
if loss is None or not math.isfinite(float(loss.detach().cpu())):
    raise RuntimeError(f"PROGRAM063_SMOKE_INVALID_LOSS={loss}")
loss.backward()
torch.cuda.synchronize()
grad_tensors = sum(
    1 for parameter in model.parameters()
    if parameter.requires_grad and parameter.grad is not None
)
if grad_tensors == 0:
    raise RuntimeError("PROGRAM063_SMOKE_NO_GRADIENTS")
trainer.optimizer.step()
trainer.optimizer.zero_grad(set_to_none=True)
model = FastLanguageModel.for_inference(model)
def cache_full_parity(ids, label):
    prefix_mask = torch.ones_like(ids)
    prefix_output_local = model(
        input_ids=ids,
        attention_mask=prefix_mask,
        return_dict=True,
        use_cache=True,
    )
    if prefix_output_local.past_key_values is None or not torch.isfinite(prefix_output_local.logits).all():
        raise RuntimeError(f"PROGRAM063_SMOKE_PREFIX_INFERENCE_FAILED label={label}")
    next_input_local = ids[:, -1:]
    next_position_local = torch.full(
        (ids.shape[0], 1),
        ids.shape[1],
        dtype=torch.long,
        device=ids.device,
    )
    cached_output = model(
        input_ids=next_input_local,
        position_ids=next_position_local,
        past_key_values=prefix_output_local.past_key_values,
        return_dict=True,
        use_cache=True,
    )
    full_ids = torch.cat([ids, next_input_local], dim=1)
    full_output = model(
        input_ids=full_ids,
        attention_mask=torch.ones_like(full_ids),
        return_dict=True,
        use_cache=False,
    )
    cached_logits = cached_output.logits[:, -1, :].float()
    full_logits = full_output.logits[:, -1, :].float()
    if cached_logits.shape != full_logits.shape:
        raise RuntimeError(
            f"PROGRAM063_SMOKE_CACHE_FULL_SHAPE label={label} cached={cached_logits.shape} full={full_logits.shape}"
        )
    if not torch.isfinite(cached_logits).all() or not torch.isfinite(full_logits).all():
        raise RuntimeError(f"PROGRAM063_SMOKE_CACHE_FULL_NONFINITE label={label}")
    delta = cached_logits - full_logits
    absolute_delta = delta.abs()
    max_abs = float(absolute_delta.max().cpu())
    mean_abs = float(absolute_delta.mean().cpu())
    rmse = float(torch.sqrt(torch.mean(delta.square())).cpu())
    reference_rms = float(torch.sqrt(torch.mean(full_logits.square())).cpu())
    relative_rmse = rmse / max(reference_rms, 1e-6)
    cosine = F.cosine_similarity(cached_logits, full_logits, dim=-1)
    cosine_min = float(cosine.min().cpu())
    top1_agreement = float(
        (cached_logits.argmax(dim=-1) == full_logits.argmax(dim=-1)).float().mean().cpu()
    )
    arc_token_ids = torch.as_tensor(ARC_TOKENS, dtype=torch.long, device=cached_logits.device)
    if int(arc_token_ids.max().item()) >= cached_logits.shape[-1]:
        raise RuntimeError(
            f"PROGRAM063_SMOKE_ARC_TOKEN_RANGE max_token={int(arc_token_ids.max().item())} "
            f"vocab={cached_logits.shape[-1]}"
        )
    cached_arc_nll = (
        torch.logsumexp(cached_logits, dim=-1, keepdim=True)
        - cached_logits.index_select(-1, arc_token_ids)
    )
    full_arc_nll = (
        torch.logsumexp(full_logits, dim=-1, keepdim=True)
        - full_logits.index_select(-1, arc_token_ids)
    )
    arc_nll_delta = (cached_arc_nll - full_arc_nll).abs()
    dfs_max_score = -math.log(0.2)
    cached_alive = cached_arc_nll < dfs_max_score
    full_alive = full_arc_nll < dfs_max_score
    threshold_mismatch_count = int((cached_alive != full_alive).sum().item())
    threshold_decision_count = int(cached_alive.numel())
    threshold_decision_agreement = 1.0 - (
        threshold_mismatch_count / max(threshold_decision_count, 1)
    )
    arc_nll_max_abs = float(arc_nll_delta.max().cpu())
    arc_nll_mean_abs = float(arc_nll_delta.mean().cpu())
    full_threshold_margin_min = float((full_arc_nll - dfs_max_score).abs().min().cpu())
    top_k = min(4, cached_arc_nll.shape[-1])
    cached_topk = torch.topk(-cached_arc_nll, k=top_k, dim=-1).indices
    full_topk = torch.topk(-full_arc_nll, k=top_k, dim=-1).indices
    topk_position_agreement = float((cached_topk == full_topk).float().mean().cpu())
    # Bfloat16 cached and full kernels need not be bit-identical. Scale-aware
    # error, cosine direction, and the production batch's argmaxes must agree.
    if max_abs > 4.0 or relative_rmse > 0.08 or cosine_min < 0.99 or top1_agreement < 0.75:
        raise RuntimeError(
            "PROGRAM063_SMOKE_CACHE_FULL_PARITY_FAILED="
            + json.dumps({
                "label": label,
                "max_abs": max_abs,
                "mean_abs": mean_abs,
                "rmse": rmse,
                "reference_rms": reference_rms,
                "relative_rmse": relative_rmse,
                "cosine_min": cosine_min,
                "top1_agreement": top1_agreement,
            }, sort_keys=True)
        )
    return prefix_output_local, cached_output, {
        "label": label,
        "max_abs": max_abs,
        "mean_abs": mean_abs,
        "rmse": rmse,
        "reference_rms": reference_rms,
        "relative_rmse": relative_rmse,
        "cosine_min": cosine_min,
        "top1_agreement": top1_agreement,
        "arc_nll_max_abs": arc_nll_max_abs,
        "arc_nll_mean_abs": arc_nll_mean_abs,
        "dfs_threshold": dfs_max_score,
        "threshold_mismatch_count": threshold_mismatch_count,
        "threshold_decision_count": threshold_decision_count,
        "threshold_decision_agreement": threshold_decision_agreement,
        "full_threshold_margin_min": full_threshold_margin_min,
        "arc_top4_position_agreement": topk_position_agreement,
    }

with torch.no_grad():
    base_prefix = batch["input_ids"][:, : min(8, batch["input_ids"].shape[1])]
    prefix_ids = base_prefix.repeat(4, 1)
    prefix_output, next_output, batch4_parity = cache_full_parity(prefix_ids, "batch4")
    _, batch1_output, batch1_parity = cache_full_parity(base_prefix[:1], "batch1")
    if next_output.logits.shape[0] != 4 or next_output.logits.shape[1] != 1:
        raise RuntimeError(f"PROGRAM063_SMOKE_CACHED_SHAPE={next_output.logits.shape}")
    if batch1_output.logits.shape[0] != 1 or batch1_output.logits.shape[1] != 1:
        raise RuntimeError(f"PROGRAM063_SMOKE_BATCH1_CACHED_SHAPE={batch1_output.logits.shape}")
print("PROGRAM063_MODEL_SMOKE_OK=" + json.dumps({
    "loss": float(loss.detach().cpu()),
    "lora_tensors": len(state),
    "gradient_tensors": grad_tensors,
    "batch_tokens": int(batch["input_ids"].numel()),
    "supervised_tokens": int((batch["labels"] != -100).sum().item()),
    "prefix_logits_shape": list(prefix_output.logits.shape),
    "cached_logits_shape": list(next_output.logits.shape),
    "batch1_cached_logits_shape": list(batch1_output.logits.shape),
    "batch4_cache_full_parity": batch4_parity,
    "batch1_cache_full_parity": batch1_parity,
    "allocated_mb": round(torch.cuda.memory_allocated(0) / 1024 / 1024, 1),
    "reserved_mb": round(torch.cuda.memory_reserved(0) / 1024 / 1024, 1),
    "device": torch.cuda.get_device_name(0),
}, sort_keys=True))
''',
    encoding="utf-8",
)
smoke_env = run_env.copy()
smoke_env.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
smoke = subprocess.run(
    [str(python_executable), str(smoke_path)],
    env=smoke_env,
    cwd="/kaggle/working",
    text=True,
    capture_output=True,
    timeout=600,
)
if smoke.stdout:
    print(smoke.stdout, end="")
if smoke.returncode != 0:
    if smoke.stderr:
        print(smoke.stderr, file=sys.stderr, end="")
    raise RuntimeError(f"PROGRAM063_MODEL_SMOKE_FAILED exit_code={smoke.returncode}")
if "PROGRAM063_SMOKE_SEED_READY=" not in smoke.stdout:
    raise RuntimeError("PROGRAM063_SMOKE_SEED_MARKER_MISSING")
if "PROGRAM063_MODEL_SMOKE_OK=" not in smoke.stdout:
    raise RuntimeError("PROGRAM063_MODEL_SMOKE_MARKER_MISSING")

command = [
    str(python_executable),
    "starter.py",
    "--end-time",
    str(run_plan["global_end_time"]),
    "--challenge-path",
    run_plan["challenge_path"],
]
print(
    "PROGRAM063_CODE_PREFLIGHT_OK "
    f"profile={run_plan['runtime_profile']} python={python_executable} "
    f"runtime_family={runtime_family} runtime_lock=pinned qwen3_attention={qwen3_probe['backend']} "
    f"exact_solver_sha256={actual_hashes['arc_solver.py']} "
    f"overlay={overlay_root}"
)
print("Launching Program No. 063:", " ".join(command))
starter_log_path = Path("/kaggle/working/program063_starter.log")
starter_lock_path = Path("/kaggle/working/program063_starter.lock")
if starter_lock_path.exists():
    stale_owner = starter_lock_path.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"pid=(\d+)", stale_owner)
    owner_alive = False
    if match:
        try:
            os.kill(int(match.group(1)), 0)
            owner_alive = True
        except (ProcessLookupError, PermissionError, ValueError):
            owner_alive = False
    if owner_alive:
        raise RuntimeError(f"PROGRAM063_EXISTING_STARTER_ALIVE={stale_owner.strip()}")
    starter_lock_path.unlink()

# Do not let the production subprocess inherit the notebook's stdout/stderr file
# descriptors. Kaggle/Papermill can display inherited child output twice. One
# PIPE is owned by this kernel, persisted verbatim, and replayed exactly once.
with starter_log_path.open("w", encoding="utf-8", buffering=1) as starter_log_handle:
    process = subprocess.Popen(
        command,
        env=run_env,
        cwd="/kaggle/working",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("PROGRAM063_STARTER_PIPE_MISSING")
    for line in process.stdout:
        starter_log_handle.write(line)
        print(line, end="", flush=True)
    return_code = process.wait()

starter_log_text = starter_log_path.read_text(encoding="utf-8", errors="replace")
starter_log_sha256 = hashlib.sha256(starter_log_text.encode("utf-8")).hexdigest()
if return_code != 0:
    tail = "\n".join(starter_log_text.splitlines()[-80:])
    raise RuntimeError(
        f"PROGRAM063_STARTER_FAILED exit_code={return_code} log={starter_log_path}\n{tail}"
    )
ready_count = starter_log_text.count("PROGRAM063_STARTER_READY")
complete_count = starter_log_text.count("PROGRAM063_STARTER_COMPLETE")
worker_start_counts = {
    str(rank): len(re.findall(rf"^\[Rank {rank}\] start! pid=", starter_log_text, flags=re.MULTILINE))
    for rank in range(4)
}
worker_done_counts = {
    str(rank): len(re.findall(rf"^\[Rank {rank}\] done! pid=", starter_log_text, flags=re.MULTILINE))
    for rank in range(4)
}
if ready_count != 1 or complete_count != 1:
    raise RuntimeError(
        "PROGRAM063_STARTER_PARENT_COUNT_MISMATCH="
        + json.dumps({"ready": ready_count, "complete": complete_count}, sort_keys=True)
    )
if any(count != 1 for count in worker_start_counts.values()):
    raise RuntimeError(
        "PROGRAM063_WORKER_START_COUNT_MISMATCH="
        + json.dumps(worker_start_counts, sort_keys=True)
    )
if any(count != 1 for count in worker_done_counts.values()):
    raise RuntimeError(
        "PROGRAM063_WORKER_DONE_COUNT_MISMATCH="
        + json.dumps(worker_done_counts, sort_keys=True)
    )
def parse_json_markers(text, marker):
    reports = []
    prefix = marker + "="
    for line in text.splitlines():
        if line.startswith(prefix):
            reports.append(json.loads(line[len(prefix):]))
    return reports

worker_seed_reports = parse_json_markers(starter_log_text, "PROGRAM063_WORKER_SEED_READY")
task_seed_reports = parse_json_markers(starter_log_text, "PROGRAM063_TASK_SEED_READY")
if len(worker_seed_reports) != 4:
    raise RuntimeError(
        "PROGRAM063_WORKER_SEED_COUNT_MISMATCH="
        + json.dumps({"count": len(worker_seed_reports), "reports": worker_seed_reports}, sort_keys=True)
    )
worker_ranks = sorted(report.get("rank") for report in worker_seed_reports)
if worker_ranks != [0, 1, 2, 3]:
    raise RuntimeError(f"PROGRAM063_WORKER_SEED_RANK_MISMATCH={worker_ranks}")
for report in worker_seed_reports + task_seed_reports:
    if report.get("base_seed") != 42 or report.get("python_hash_seed") != "0":
        raise RuntimeError("PROGRAM063_SEED_VALUE_MISMATCH=" + json.dumps(report, sort_keys=True))
    if report.get("seed_policy") != "rng_only_no_backend_change":
        raise RuntimeError("PROGRAM063_SEED_POLICY_MISMATCH=" + json.dumps(report, sort_keys=True))
    if report.get("deterministic_algorithms_enabled") is not False:
        raise RuntimeError("PROGRAM063_BACKEND_POLICY_CHANGED=" + json.dumps(report, sort_keys=True))

hash_probes = {report.get("hash_probe") for report in worker_seed_reports + task_seed_reports}
if len(hash_probes) != 1:
    raise RuntimeError(f"PROGRAM063_HASH_PROBE_MISMATCH={sorted(hash_probes)}")

if not run_plan["rerun_mode"]:
    expected_tasks = {"0934a4d8", "36a08778", "981571dc", "aa4ec2a5"}
    actual_tasks = {report.get("task_id") for report in task_seed_reports}
    if len(task_seed_reports) != 4 or actual_tasks != expected_tasks:
        raise RuntimeError(
            "PROGRAM063_TASK_SEED_COVERAGE_MISMATCH="
            + json.dumps({
                "count": len(task_seed_reports),
                "expected": sorted(expected_tasks),
                "actual": sorted(task for task in actual_tasks if task is not None),
            }, sort_keys=True)
        )

training_pattern = re.compile(
    r"^\[Rank (?P<rank>\d+)\] training stats for puzzle (?P<task>[0-9a-f]+): "
    r"TrainOutput\(global_step=(?P<global_step>\d+), training_loss=(?P<training_loss>[-+0-9.eE]+)",
    flags=re.MULTILINE,
)
training_records = []
for match in training_pattern.finditer(starter_log_text):
    training_records.append({
        "rank": int(match.group("rank")),
        "task_id": match.group("task"),
        "global_step": int(match.group("global_step")),
        "training_loss": float(match.group("training_loss")),
    })
if not run_plan["rerun_mode"] and len(training_records) != 4:
    raise RuntimeError(
        "PROGRAM063_TRAINING_RECORD_COUNT_MISMATCH="
        + json.dumps(training_records, sort_keys=True)
    )

diagnostics = {
    "program_no": 63,
    "seed_contract": run_plan["seed_contract"],
    "starter_log": str(starter_log_path),
    "starter_log_sha256": starter_log_sha256,
    "worker_seed_reports": sorted(worker_seed_reports, key=lambda item: item["rank"]),
    "task_seed_reports": sorted(
        task_seed_reports,
        key=lambda item: (str(item.get("task_id")), int(item.get("rank", -1))),
    ),
    "training_records": sorted(training_records, key=lambda item: item["task_id"]),
}
diagnostics_path = Path("/kaggle/working/program063_run_diagnostics.json")
diagnostics_path.write_text(
    json.dumps(diagnostics, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)
diagnostics_sha256 = hashlib.sha256(diagnostics_path.read_bytes()).hexdigest()
print("PROGRAM063_SEED_AUDIT_OK=" + json.dumps({
    "worker_seed_count": len(worker_seed_reports),
    "task_seed_count": len(task_seed_reports),
    "hash_probe": next(iter(hash_probes)),
    "base_seed": 42,
    "python_hash_seed": "0",
    "deterministic_algorithms_enabled": False,
    "backend_policy_changed": False,
}, sort_keys=True))
print(f"PROGRAM063_RUN_DIAGNOSTICS_SHA256={diagnostics_sha256}")
print("PROGRAM063_TRAINING_RECORDS=" + json.dumps(training_records, sort_keys=True))

print("PROGRAM063_SINGLE_PIPE_LOGGING_OK=" + json.dumps({
    "starter_log": str(starter_log_path),
    "starter_log_sha256": starter_log_sha256,
    "ready_count": ready_count,
    "complete_count": complete_count,
    "worker_start_counts": worker_start_counts,
    "worker_done_counts": worker_done_counts,
    "relay_owner_pid": os.getpid(),
}, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

run_plan = json.loads(Path("program063_run_plan.json").read_text(encoding="utf-8"))
legacy_root = Path(run_plan["legacy_root"]).resolve()
overlay_root = Path(run_plan["overlay_root"]).resolve()
python_executable = Path(run_plan["python_executable"]).resolve()

finalize_path = Path("/kaggle/working/program063_finalize.py")
finalize_path.write_text(
    '''import os
import sys
from pathlib import Path
legacy_root = Path(os.environ["PROGRAM063_LEGACY_ROOT"]).resolve()
overlay_root = Path(os.environ["PROGRAM063_OVERLAY_ROOT"]).resolve()
clean = []
for item in sys.path:
    if not item:
        clean.append(item)
        continue
    try:
        resolved = Path(item).resolve()
    except OSError:
        continue
    if resolved == legacy_root or legacy_root in resolved.parents or resolved == overlay_root:
        continue
    clean.append(item)
sys.path[:] = [str(overlay_root)] + clean
'''
    + r'''
import hashlib
import json
import os
import re
import sys
from pathlib import Path

import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, hashable, score_kgmon, score_full_probmul_3


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _origin_name(sample_key: str) -> str:
    return re.sub(r"\.out\d+$", "", sample_key)


def _geometry_family(sample_key: str) -> str:
    # Ignore color permutation and example-order suffixes; keep only D4 geometry.
    origin = _origin_name(sample_key)
    parts = origin.split(".")[1:]
    prefix = []
    for part in parts:
        if part.startswith("permute"):
            break
        if part.startswith("ex") or part.startswith("out") or part.startswith("run"):
            continue
        prefix.append(part)
    transposed = "transpose" in prefix
    rotations = sum(part == "rot90" for part in prefix) % 4
    return ("transpose_" if transposed else "plain_") + f"rot{90 * rotations}"


def _candidate_metrics(basevalues: dict) -> dict:
    grouped = {}
    for sample_key, sample in basevalues.items():
        h = hashable(sample["solution"])
        row = grouped.setdefault(h, {
            "solution": sample["solution"],
            "sample_keys": [],
            "origins": set(),
            "families": set(),
            "beam_scores": [],
            "aug_means": [],
        })
        row["sample_keys"].append(sample_key)
        row["origins"].add(_origin_name(sample_key))
        row["families"].add(_geometry_family(sample_key))
        row["beam_scores"].append(float(sample["beam_score"]))
        row["aug_means"].append(float(np.mean(sample["score_aug"])))

    guesses = {k: g for k, g in basevalues.items()}
    kgmon = score_kgmon(guesses)
    probmul = score_full_probmul_3(guesses)
    kg_rank = {hashable(solution): i + 1 for i, solution in enumerate(kgmon)}
    prob_rank = {hashable(solution): i + 1 for i, solution in enumerate(probmul)}

    for h, row in grouped.items():
        row["raw_support_count"] = len(row["sample_keys"])
        row["origin_count"] = len(row["origins"])
        row["family_count"] = len(row["families"])
        row["families"] = sorted(row["families"])
        row["mean_beam_score"] = float(np.mean(row["beam_scores"])) if row["beam_scores"] else float("inf")
        row["mean_aug_score"] = float(np.mean(row["aug_means"])) if row["aug_means"] else float("inf")
        row["kgmon_rank"] = int(kg_rank.get(h, 10**9))
        row["probmul_rank"] = int(prob_rank.get(h, 10**9))
    return grouped


def _public_metrics(row):
    if row is None:
        return None
    return {
        "family_count": row["family_count"],
        "origin_count": row["origin_count"],
        "raw_support_count": row["raw_support_count"],
        "kgmon_rank": row["kgmon_rank"],
        "probmul_rank": row["probmul_rank"],
        "mean_beam_score": row["mean_beam_score"],
        "mean_aug_score": row["mean_aug_score"],
        "families": row["families"],
    }


def conservative_cross_symmetry_two_guess(basevalues: dict):
    # Keep KGMon top-1 exactly; only a narrowly-qualified candidate may replace rank 2.
    guesses = {k: g for k, g in basevalues.items()}
    kgmon = score_kgmon(guesses)
    if not kgmon:
        return [], {"changed": False, "reason": "NO_CANDIDATES", "top1_preserved": True,
                    "baseline_second": None, "selected_second": None}
    if len(kgmon) == 1:
        return [kgmon[0]], {"changed": False, "reason": "ONLY_ONE_UNIQUE_CANDIDATE",
                            "top1_preserved": True, "baseline_second": None, "selected_second": None}

    metrics = _candidate_metrics(basevalues)
    top1 = kgmon[0]
    baseline_second = kgmon[1]
    top1_h = hashable(top1)
    baseline_h = hashable(baseline_second)
    baseline_row = metrics[baseline_h]

    qualified = []
    for h, row in metrics.items():
        if h == top1_h or h == baseline_h:
            continue
        if row["kgmon_rank"] > 4 or row["probmul_rank"] > 4:
            continue
        if row["family_count"] < 2 or row["origin_count"] < 2:
            continue
        if row["family_count"] <= baseline_row["family_count"]:
            continue
        if row["origin_count"] < baseline_row["origin_count"]:
            continue
        qualified.append((h, row))

    if not qualified:
        return [top1, baseline_second], {
            "changed": False,
            "reason": "NO_STRICTLY_BROADER_TOP4_CANDIDATE",
            "top1_preserved": True,
            "baseline_second": _public_metrics(baseline_row),
            "selected_second": _public_metrics(baseline_row),
        }

    qualified.sort(key=lambda item: (
        -item[1]["family_count"],
        -item[1]["origin_count"],
        item[1]["kgmon_rank"] + item[1]["probmul_rank"],
        item[1]["kgmon_rank"],
        item[1]["probmul_rank"],
        item[1]["mean_aug_score"],
        item[1]["mean_beam_score"],
    ))
    selected_h, selected_row = qualified[0]
    selected_second = selected_row["solution"]
    return [top1, selected_second], {
        "changed": True,
        "reason": "PROMOTED_STRICTLY_BROADER_CROSS_SYMMETRY_SUPPORT",
        "top1_preserved": True,
        "baseline_second": _public_metrics(baseline_row),
        "selected_second": _public_metrics(selected_row),
    }


legacy_root = Path(os.environ["PROGRAM063_LEGACY_ROOT"]).resolve()
overlay_root = Path(os.environ["PROGRAM063_OVERLAY_ROOT"]).resolve()
numpy_path = Path(np.__file__).resolve()
if numpy_path == legacy_root or legacy_root in numpy_path.parents:
    raise RuntimeError(f"Finalize NumPy leaked from legacy root: {numpy_path}")
if numpy_path == overlay_root or overlay_root in numpy_path.parents:
    raise RuntimeError(f"Finalize NumPy leaked from overlay: {numpy_path}")
print("PROGRAM063_FINALIZE_RUNTIME_OK=" + json.dumps({"numpy_file": str(numpy_path), "python": sys.executable}, sort_keys=True))

run_plan = json.loads(Path("program063_run_plan.json").read_text(encoding="utf-8"))
challenge_path = Path(run_plan["challenge_path"])
rerun_mode = bool(run_plan["rerun_mode"])

data = ArcDataset.from_file(str(challenge_path))
if not rerun_mode:
    solution_path = Path(run_plan["solution_path"])
    data = data.load_replies(str(solution_path))

output_root = Path("/kaggle/inference_outputs")
if not output_root.is_dir():
    raise RuntimeError(f"PROGRAM063_OUTPUT_DIRECTORY_MISSING={output_root}")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results(str(output_root))

baseline_results = decoder.run_selection_algo(score_kgmon)
guarded_results = {}
change_details = {}
for basekey, basevalues in sorted(decoder.decoded_results.items()):
    selected, detail = conservative_cross_symmetry_two_guess(basevalues)
    guarded_results[basekey] = selected
    change_details[basekey] = detail

for basekey in sorted(baseline_results):
    baseline = baseline_results[basekey]
    guarded = guarded_results.get(basekey, [])
    if baseline and guarded and hashable(baseline[0]) != hashable(guarded[0]):
        raise RuntimeError(f"PROGRAM063_TOP1_SAFETY_INVARIANT_FAILED={basekey}")

baseline_submission = data.get_submission(baseline_results)
guarded_submission = data.get_submission(guarded_results)

baseline_path = Path("/kaggle/working/submission_program063_baseline_kgmon.json")
baseline_path.write_text(json.dumps(baseline_submission, sort_keys=True, separators=(",", ":")), encoding="utf-8")
submission_path = Path("/kaggle/working/submission.json")
submission_path.write_text(json.dumps(guarded_submission, sort_keys=True, separators=(",", ":")), encoding="utf-8")
if not submission_path.is_file() or submission_path.stat().st_size <= 0:
    raise RuntimeError(f"PROGRAM063_SUBMISSION_JSON_NOT_WRITTEN={submission_path}")
print("PROGRAM063_SUBMISSION_JSON_READY=" + json.dumps({
    "path": str(submission_path), "size": submission_path.stat().st_size,
    "shadow_baseline_path": str(baseline_path)}, sort_keys=True))

output_files = sorted(path for path in output_root.iterdir() if path.is_file())
output_entries = []
for path in output_files:
    task_id = path.name.split("_", 1)[0]
    output_entries.append({"task_id": task_id, "name": path.name, "size": path.stat().st_size, "sha256": sha256_file(path)})
manifest_tasks = {}
for entry in output_entries:
    manifest_tasks.setdefault(entry["task_id"], []).append(entry)
program024_reference_total = 76 if not rerun_mode else None
program024_delta = len(output_entries) - program024_reference_total if program024_reference_total is not None else None

diagnostics_path = Path("program063_run_diagnostics.json")
if not diagnostics_path.is_file():
    raise RuntimeError(f"Program 063 diagnostics are missing: {diagnostics_path}")
diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
diagnostics_sha256 = sha256_file(diagnostics_path)

changed_keys = sorted(k for k, v in change_details.items() if v.get("changed"))
compact_change_details = {k: change_details[k] for k in changed_keys}
ranker_audit = {
    "policy": "keep_kgmon_top1_replace_second_only_on_strictly_broader_cross_symmetry_top4_support",
    "top1_changed_count": 0,
    "second_slot_changed_count": len(changed_keys),
    "changed_keys": changed_keys,
    "change_details": compact_change_details,
}
ranker_audit_path = Path("program063_ranker_audit.json")
ranker_audit_path.write_text(json.dumps(ranker_audit, sort_keys=True, separators=(",", ":")), encoding="utf-8")

manifest = {
    "program_no": 63,
    "variant": "exact024_cross_symmetry_second_guess_guard",
    "rerun_mode": rerun_mode,
    "seed_contract": run_plan["seed_contract"],
    "run_diagnostics_sha256": diagnostics_sha256,
    "training_records": diagnostics.get("training_records", []),
    "total_files": len(output_entries),
    "program024_reference_total": program024_reference_total,
    "delta_from_program024_reference": program024_delta,
    "ranker_audit_sha256": sha256_file(ranker_audit_path),
    "tasks": manifest_tasks,
}
manifest_path = Path("program063_output_manifest.json")
manifest_path.write_text(json.dumps(manifest, sort_keys=True, separators=(",", ":")), encoding="utf-8")
print(f"PROGRAM063_OUTPUT_FILE_COUNT={len(output_entries)}")
if program024_delta is None:
    print("PROGRAM063_OUTPUT_COUNT_DELTA_FROM_PROGRAM024=NA_RERUN_MODE")
else:
    print(f"PROGRAM063_OUTPUT_COUNT_DELTA_FROM_PROGRAM024={program024_delta}")
print(f"PROGRAM063_OUTPUT_MANIFEST_SHA256={sha256_file(manifest_path)}")
print("PROGRAM063_RANKER_AUDIT=" + json.dumps(ranker_audit, sort_keys=True))
for task_id in sorted(manifest_tasks):
    entries = manifest_tasks[task_id]
    combined = hashlib.sha256()
    for entry in entries:
        combined.update(f"{entry['name']}:{entry['sha256']}\n".encode("utf-8"))
    print("PROGRAM063_TASK_OUTPUT_MANIFEST=" + json.dumps({
        "task_id": task_id, "count": len(entries), "combined_sha256": combined.hexdigest(),
        "files": [entry["name"] for entry in entries]}, sort_keys=True))

submission_sha256 = sha256_file(submission_path)
baseline_sha256 = sha256_file(baseline_path)
print(f"PROGRAM063_SUBMISSION_SHA256={submission_sha256}")
print(f"PROGRAM063_SHADOW_BASELINE_SUBMISSION_SHA256={baseline_sha256}")

reproducibility_payload = {
    "program_no": 63,
    "variant": "exact024_cross_symmetry_second_guess_guard",
    "seed_contract": run_plan["seed_contract"],
    "runtime_tree_sha256": {name: report["record_tree_sha256"] for name, report in sorted(run_plan["pinned_runtime_source"].items())},
    "exact_program024_source_sha256": {
        "arc_loader.py": "d01cd56167e534ae156706ef62ab11662fc7b403964614a7a55131940bde970c",
        "arc_decoder.py": "965cfd910777d9bbec9681c6c15c5e2ea3924569734248046f5e58f16cb28222",
        "arc_solver.py": "f9011b4d4549fb688488d67ba7c2dc43184824436d34e1a4c2eae06f69daab73",
    },
    "run_diagnostics_sha256": diagnostics_sha256,
    "output_manifest_sha256": sha256_file(manifest_path),
    "ranker_audit_sha256": sha256_file(ranker_audit_path),
    "shadow_baseline_submission_sha256": baseline_sha256,
    "submission_sha256": submission_sha256,
}
reproducibility_path = Path("program063_reproducibility_signature.json")
reproducibility_path.write_text(json.dumps(reproducibility_payload, sort_keys=True, separators=(",", ":")), encoding="utf-8")
reproducibility_sha256 = sha256_file(reproducibility_path)
print(f"PROGRAM063_REPRODUCIBILITY_SIGNATURE={reproducibility_sha256}")

baseline_public_score = "NA_COMPETITION_RERUN"
guarded_public_score = "NA_COMPETITION_RERUN"
if rerun_mode:
    print("PROGRAM063_SUBMISSION_BUILD_COMPLETE")
    print("SUBMISSION_READY_FOR_MANUAL_SUBMIT")
else:
    decoder.benchmark_selection_algos()
    baseline_public_score = data.validate_submission(json.loads(baseline_path.read_text(encoding="utf-8")))
    guarded_public_score = data.validate_submission(json.loads(submission_path.read_text(encoding="utf-8")))
    print(f"PROGRAM063_BASELINE_PUBLIC_RELOAD_SCORE={baseline_public_score}")
    print(f"PROGRAM063_GUARDED_PUBLIC_RELOAD_SCORE={guarded_public_score}")
    print("PROGRAM063_PUBLIC_EVAL_COMPLETE_REVIEW_RANKER_EFFECT_BEFORE_SUBMITTING")

task_counts = {task_id: len(entries) for task_id, entries in sorted(manifest_tasks.items())}
gpt_fb_lines = [
    "＝＝＝＝GPT FB START ＝＝＝＝",
    "GPT_FB_VERSION: 7",
    "PROGRAM_NO: 063",
    "PURPOSE: Preserve exact Program 024 candidate generation and KGMon top-1 while replacing only a weak second guess when a nearby candidate has strictly broader cross-symmetry support.",
    "EXECUTION_STATUS: COMPLETED",
    "FIRST_ERROR: NONE",
    "SUBMIT_POLICY: EXPERIMENTAL_SECOND_SLOT_ONLY_TOP1_EXACTLY_PRESERVED_REVIEW_PUBLIC_COMPARISON_BEFORE_SUBMIT",
    f"RUNTIME_FAMILY: {run_plan.get('runtime_family')}",
    "EXTERNAL_ASSETS: NONE_BEYOND_EXISTING_PROGRAM024_STACK",
    "SOURCE_HASH_CHECK: EXACT_PROGRAM024_OK",
    "SEED_AUDIT: OK",
    "BACKEND_POLICY: NORMAL_NONDETERMINISTIC_CUDA_BACKWARD",
    "RANKER_POLICY: KEEP_KGMON_TOP1_PROMOTE_SECOND_ONLY_IF_TOP4_CANDIDATE_HAS_STRICTLY_BROADER_GEOMETRY_FAMILY_SUPPORT_AND_RANK_CONSENSUS",
    "TOP1_CHANGED_COUNT: 0",
    f"SECOND_SLOT_CHANGED_COUNT: {len(changed_keys)}",
    "SECOND_SLOT_CHANGED_KEYS: " + json.dumps(changed_keys, sort_keys=True),
    "SECOND_SLOT_CHANGE_DETAILS: " + json.dumps(compact_change_details, sort_keys=True),
    "TRAINING_RECORDS: " + json.dumps(diagnostics.get("training_records", []), sort_keys=True),
    f"OUTPUT_FILE_COUNT: {len(output_entries)}",
    "TASK_OUTPUT_COUNTS: " + json.dumps(task_counts, sort_keys=True),
    f"OUTPUT_MANIFEST_SHA256: {sha256_file(manifest_path)}",
    f"SHADOW_BASELINE_SUBMISSION_SHA256: {baseline_sha256}",
    f"SUBMISSION_PATH: {submission_path}",
    f"SUBMISSION_SHA256: {submission_sha256}",
    f"REPRODUCIBILITY_SIGNATURE: {reproducibility_sha256}",
    f"BASELINE_PUBLIC_RELOAD_SCORE: {baseline_public_score}",
    f"GUARDED_PUBLIC_RELOAD_SCORE: {guarded_public_score}",
    "DIAGNOSTIC_CONCLUSION: EXACT_TOP1_PRESERVED_AND_CROSS_SYMMETRY_SECOND_GUESS_GUARD_APPLIED_WITH_ZERO_EXTRA_GPU_PASSES",
    "NEXT_RECOMMENDED_ACTION: SUBMIT_AT_MOST_ONCE_ONLY_IF_RUN_COMPLETES_AND_GUARDED_PUBLIC_SCORE_DOES_NOT_REGRESS_VERSUS_SHADOW_BASELINE",
    "PRIVATE_SCORE: USER_TO_FILL",
    "＝＝＝＝GPT FB STOP ＝＝＝＝",
]
print("\n".join(gpt_fb_lines))
''',
    encoding="utf-8",
)

finalize_env = os.environ.copy()
finalize_env.update({
    "PROGRAM063_LEGACY_ROOT": str(legacy_root),
    "PROGRAM063_OVERLAY_ROOT": str(overlay_root),
    "PYTHONPATH": run_plan["runtime_pythonpath"],
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": run_plan["ptxas_path"],
    "PYTHONHASHSEED": run_plan["seed_contract"]["python_hash_seed"],
    "PROGRAM063_BASE_SEED": str(run_plan["seed_contract"]["base_seed"]),
    "PROGRAM063_SEED_POLICY": "rng_only_no_backend_change",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})
finalize = subprocess.run(
    [str(python_executable), str(finalize_path)],
    env=finalize_env,
    cwd="/kaggle/working",
    text=True,
    capture_output=True,
)
if finalize.stdout:
    print(finalize.stdout, end="")
if finalize.returncode != 0:
    if finalize.stderr:
        print(finalize.stderr, file=sys.stderr, end="")
    raise RuntimeError(f"PROGRAM063_FINALIZE_FAILED exit_code={finalize.returncode}")
